# Group 6 — LLM-primary Fast9 pipeline

## Team walkthrough version

This notebook implements the second Group 6 submission: an **LLM-primary, target-specific survey-response pipeline**. It predicts nine hidden World Values Survey answers for every respondent.

The model makes one separate prediction for each respondent–target pair:

\[
\text{one respondent} \times 9 \text{ targets} = 9 \text{ LLM calls}.
\]

For the official test set of 1,050 respondents, this produces:

\[
1{,}050 \times 9 = 9{,}450
\]

primary LLM requests.

## High-level data flow

```text
Official train / test / targets / features
                    ↓
Decode survey metadata and official target labels
                    ↓
Create leakage-safe validation splits
                    ↓
For each target:
    NMI relevance ranking
    → mRMR redundancy reduction
    → CatBoost model and feature importance
    → hybrid feature ranking
                    ↓
For each respondent–target pair:
    direct respondent answers
    + two retrieved labelled analogues
    + CatBoost probability prior
    + target-specific reasoning guide
                    ↓
Qwen/Qwen3-32B returns probabilities
                    ↓
88% LLM probabilities + 12% CatBoost probabilities
                    ↓
Final official label
```

## What makes this an LLM-primary pipeline?

The LLM is the main prediction component whenever it returns a valid response. CatBoost is used to:

1. help select useful nonlinear features;
2. provide an auxiliary probability prior;
3. contribute a small 12% probability blend;
4. guarantee coverage if both the initial LLM response and the repair response fail.

On validation, the repair and fallback rates were zero, so all predictions contained a valid LLM component.

## How to use this notebook during a team review

Each code cell is preceded by an explanation covering:

- **Purpose** — why the cell exists;
- **What happens** — the main operations in plain language;
- **Main outputs** — objects created for later cells;
- **API / cost impact** — whether the cell sends paid LLM requests;
- **What to check** — expected outputs or possible failure points.

The code is unchanged from the English notebook. Only the documentation has been expanded.

## Block 1. Install the required libraries

### Purpose

This cell installs the external Python packages used by the notebook. Google Colab already includes many general-purpose packages, but the challenge-specific pipeline needs several additional libraries.

### What happens

The command installs:

- `datasets` — downloads the official challenge datasets from Hugging Face;
- `openai` — provides an OpenAI-compatible client for the Nebius API;
- `numpy` — numerical operations;
- `pandas` — tables and data manipulation;
- `scikit-learn` — NMI, Macro-F1 and accuracy calculations;
- `catboost` — the auxiliary categorical prediction models;
- `tqdm` — progress bars.

The `-q` flag makes installation quieter.

### Main outputs

No Python object is created. The packages become available to later cells.

### API / cost impact

No LLM request is made. There is no Nebius cost.

### What to check

The cell should finish without a red error message. A runtime restart is usually unnecessary in Colab, but if an import later fails, rerun this cell and then rerun the imports cell.

In [1]:
%pip install -q datasets openai numpy pandas scikit-learn catboost tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.7 MB/s eta 0:00:00


## Block 2. Import modules and set display options

### Purpose

This cell imports every standard-library and third-party component used in the pipeline.

### What happens

The imports support several distinct parts of the workflow:

- **Data and modelling:** `numpy`, `pandas`, `datasets`, scikit-learn metrics;
- **LLM access:** `OpenAI`;
- **Parallel execution:** `ThreadPoolExecutor`, `as_completed`;
- **Configuration:** `dataclass`, `asdict`, `field`;
- **Caching and reproducibility:** `hashlib`, `json`, `Path`, `threading`;
- **File handling:** `zipfile`, `shutil`;
- **Timing and progress:** `time`, `tqdm`;
- **Secure key entry:** `getpass`.

The cell also changes pandas display settings so that long questions, prompts and feature tables are easier to inspect.

### Main outputs

Imported names become available globally. No model is fitted and no data are loaded yet.

### API / cost impact

No API call and no paid inference.

### What to check

The cell should complete silently. If `OpenAI` cannot be imported, the variable is temporarily set to `None`; Block 5 will then raise a clear installation error.

In [2]:
from __future__ import annotations

import copy
import hashlib
import json
import math
import os
import random
import re
import shutil
import threading
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict, dataclass, field
from getpass import getpass
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
from IPython.display import display
import pandas as pd
from datasets import load_dataset
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
from sklearn.metrics import accuracy_score, f1_score, normalized_mutual_info_score
from tqdm.auto import tqdm

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

import zipfile
from datetime import datetime


## Block 3. Central configuration

### Purpose

This cell stores the main pipeline settings in one dataclass, `FastNineCallConfig`. Centralising the settings makes the experiment reproducible and allows us to create independent validation and official-run configurations later.

### What happens

The configuration is divided into six groups.

#### 1. Dataset and split settings

- repository name;
- random seed;
- number of respondents and countries used in the local in-domain and OOD splits.

#### 2. Leakage-safe feature selection

- minimum feature availability;
- minimum number of paired observations;
- size of the initial candidate pool;
- number of mRMR features;
- redundancy penalty;
- penalties for the geographic region feature in OOD and mixed modes.

#### 3. Hybrid feature plan

The selector, CatBoost importance and availability are combined to form a compact target-specific feature set.

Key values:

- 18 initial mRMR features;
- 16 hybrid features;
- 12 direct answers shown in the prompt;
- 16 features used for retrieval;
- importance weights of 70% / 20% / 10%.

#### 4. Retrieval

- two labelled analogue respondents;
- at most one same-country example;
- at most one example from each target label;
- three answers displayed per retrieved example.

#### 5. CatBoost and LLM settings

- CatBoost depth, learning rate and iterations;
- 12% CatBoost blend;
- `Qwen/Qwen3-32B`;
- temperature zero;
- extended thinking disabled;
- 48 parallel workers;
- retry and timeout settings.

#### 6. Persistence

- JSONL cache path;
- checkpoint interval.

### Main outputs

- `FastNineCallConfig` — the configuration class;
- `CFG` — the active default configuration.

The final line displays every parameter in a readable table.

### API / cost impact

No API call.

### What to check

During team review, this is the main cell for discussing modelling choices. The most consequential parameters are:

- `prompt_features`;
- `n_retrieval_examples`;
- `catboost_blend_weight`;
- `max_workers`;
- feature-selection weights and penalties.

Changing any modelling parameter creates a new model version and should ideally be followed by a new validation run.

In [3]:
@dataclass
class FastNineCallConfig:
    repo: str = "oxford-llms/ai-respondents-challenge"
    seed: int = 42

    # Exact split settings used in the saved OOD pilot
    in_domain_val_per_country: int = 30
    out_domain_n_countries: int = 15
    out_domain_val_per_country: int = 30

    # Leakage-safe selector
    min_availability: float = 0.25
    min_paired_rows: int = 100
    selection_pool_size: int = 40
    selected_features: int = 18
    redundancy_lambda: float = 0.20
    ood_region_penalty: float = 0.60
    mixed_region_penalty: float = 0.80

    # Hybrid feature-importance plan
    catboost_pool_features: int = 28
    protected_mrmr_features: int = 8
    catboost_importance_features: int = 4
    hybrid_selected_features: int = 16
    prompt_features: int = 12
    retrieval_features: int = 16
    selector_importance_weight: float = 0.70
    catboost_importance_weight: float = 0.20
    availability_weight: float = 0.10

    # Compact RAG
    use_retrieval: bool = True
    n_retrieval_examples: int = 2
    n_same_country_examples: int = 1
    max_examples_per_label: int = 1
    example_features: int = 3

    # CatBoost is an auxiliary expert, not the main method
    use_catboost: bool = True
    catboost_iterations: int = 180
    catboost_depth: int = 6
    catboost_learning_rate: float = 0.08
    catboost_in_prompt: bool = True
    catboost_blend_weight: float = 0.12

    # LLM
    model: str = "Qwen/Qwen3-32B"
    base_url: str = "https://api.studio.nebius.com/v1/"
    temperature: float = 0.0
    first_pass_max_tokens: int = 220
    repair_max_tokens: int = 220
    max_workers: int = 48
    retries: int = 2
    request_timeout_seconds: float = 120.0

    # Persistence
    cache_path: str = "llm_cache_fast9_validation.jsonl"
    checkpoint_every: int = 100


CFG = FastNineCallConfig()
display(pd.DataFrame([asdict(CFG)]).T.rename(columns={0: "value"}))


,value
repo,oxford-llms/ai-respondents-challenge
seed,42
in_domain_val_per_country,30
out_domain_n_countries,15
out_domain_val_per_country,30
min_availability,0.25
min_paired_rows,100
selection_pool_size,40
selected_features,18
redundancy_lambda,0.2


## Block 4. Load the official challenge datasets

### Purpose

This cell downloads the four official data tables and checks that the training data contain all target columns.

### What happens

It loads:

- `train` — 5,000 labelled respondents;
- `test` — 1,050 respondents whose nine target answers must be predicted;
- `targets` — target questions, answer options and exact official labels;
- `features` — the permitted feature-variable metadata and code-to-text mappings.

The cell prints:

- table dimensions;
- number of target questions;
- number of allowed features;
- number of countries in train and test.

It then verifies that every target listed in `targets` exists as a column in `train`.

### Main outputs

Four pandas DataFrames:

- `train`;
- `test`;
- `targets`;
- `features`.

### API / cost impact

This accesses Hugging Face to download data, but it does not call the LLM and does not use the Nebius budget.

### What to check

Expected broad structure:

- train contains respondent ID, country, allowed survey variables and the nine target columns;
- test contains respondent ID, country and allowed survey variables;
- targets contains nine unique `question_id` values;
- features contains the allowed feature pool.

A missing-target error indicates a dataset/version mismatch and should be resolved before continuing.

In [4]:
REPO = CFG.repo

train = load_dataset(REPO, "train", split="train").to_pandas()
test = load_dataset(REPO, "test", split="train").to_pandas()
targets = load_dataset(REPO, "targets", split="train").to_pandas()
features = load_dataset(REPO, "features", split="train").to_pandas()

print(f"Repository: {REPO}")
print(
    f"train {train.shape}, test {test.shape}, "
    f"{targets['question_id'].nunique()} targets, "
    f"{len(features)} allowed features"
)
print("train countries:", train["country"].nunique())
print("test countries:", test["country"].nunique())

required_target_columns = set(targets["question_id"].astype(str).unique())
missing_train_targets = required_target_columns - set(train.columns)
if missing_train_targets:
    raise ValueError(
        "Training data are missing target columns: "
        + ", ".join(sorted(missing_train_targets))
    )


README.md:   0%|          | 0.00/4.41k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/3.43M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

test.csv:   0%|          | 0.00/628k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

targets.csv:   0%|          | 0.00/5.02k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

features.csv:   0%|          | 0.00/191k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Repository: oxford-llms/ai-respondents-challenge
train (5000, 345), test (1050, 280), 9 targets, 278 allowed features
train countries: 50
test countries: 35


## Block 5. Create the Nebius API client

### Purpose

This cell authenticates with Nebius and creates the OpenAI-compatible client used for all Qwen requests.

### What happens

1. It confirms that the `openai` package is available.
2. It looks for the API key in the environment variable `NEBIUS_API_KEY`.
3. If no environment variable exists, Colab asks for the key securely through `getpass`.
4. The key is stored in the current runtime environment.
5. An `OpenAI` client is created with:
   - the Nebius base URL;
   - the request timeout from `CFG`;
   - automatic SDK retries disabled, because retries are handled explicitly later.

### Main outputs

- `nebius_key`;
- `raw_nebius_client`.

This is the low-level client wrapped by the cache-aware client in Block 13.

### API / cost impact

Creating the client does **not** send an inference request and does not consume tokens.

### What to check

The output should be:

```text
Nebius client created.
```

Never print, save or commit the API key. If authentication later fails, confirm that the key is valid and has sufficient balance.

In [6]:
if OpenAI is None:
    raise ImportError("Install the openai package first.")

nebius_key = os.getenv("NEBIUS_API_KEY", "").strip()
if not nebius_key:
    nebius_key = getpass("Nebius API key: ").strip()
if not nebius_key:
    raise ValueError("Nebius API key is empty.")

os.environ["NEBIUS_API_KEY"] = nebius_key

raw_nebius_client = OpenAI(
    base_url=CFG.base_url,
    api_key=nebius_key,
    timeout=CFG.request_timeout_seconds,
    max_retries=0,
)

print("Nebius client created.")


Nebius API key: ··········
Nebius client created.


## Block 6. Build survey metadata and the allowed feature pool

### Purpose

The raw challenge tables store survey answers as numeric codes. This cell prepares the metadata needed to turn those codes into readable text and defines which variables may be considered as predictors.

### What happens

#### Code normalisation

`canonical_code()` converts equivalent values to one stable string representation. For example:

- integer `1`;
- float `1.0`;
- string `"1"`,

all become `"1"`.

This is important because JSON mappings and pandas columns may represent the same code in different numeric formats.

#### Parsing answer maps

`parse_values_json()` reads the `values_json` field from the official `features` table. It includes a fallback parser in case a mapping is not perfectly formatted JSON.

#### Metadata dictionaries

The cell builds:

- `qtext` — feature code → question text;
- `vmaps` — feature code → code-to-answer-text mapping;
- `question_for` — target ID → target question text;
- `labels_for` — target ID → ordered official labels;
- `target_ids` and `target_id_set`.

#### Allowed feature restriction

`allowed_features` contains only variables that:

1. are listed in the official `features` table;
2. exist in both train and test.

`candidate_features` then removes all target variables from that list.

The final assertion protects against target leakage.

### Main outputs

The metadata dictionaries and the candidate feature list used throughout feature selection, CatBoost, retrieval and prompt construction.

### API / cost impact

No LLM request.

### What to check

The printed target list should contain nine IDs. The number of candidate features should be close to the official allowed pool after excluding targets or unavailable columns.

The final assertion must pass. If it fails, the pipeline would be attempting to use a target answer as an input.

In [7]:
def canonical_code(value: Any) -> Optional[str]:
    if pd.isna(value):
        return None
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    if isinstance(value, (np.floating, float)):
        number = float(value)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
        return format(number, "g")
    text = str(value).strip()
    try:
        number = float(text)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
        return format(number, "g")
    except (TypeError, ValueError):
        return text


def numeric_code(code: str):
    try:
        return float(code)
    except (TypeError, ValueError):
        return None


def sort_codes(codes: Iterable[str]):
    return sorted(
        codes,
        key=lambda value: (
            numeric_code(value) is None,
            numeric_code(value) if numeric_code(value) is not None else value,
        ),
    )


def parse_values_json(value: Any) -> dict[str, str]:
    if isinstance(value, dict):
        parsed = value
    elif pd.isna(value):
        parsed = {}
    else:
        try:
            parsed = json.loads(str(value))
        except json.JSONDecodeError:
            text = str(value)
            parsed = {}
            pair_pattern = re.compile(
                r'"((?:\\.|[^"\\])*)"\s*:\s*"((?:\\.|[^"\\])*)"'
            )
            for match in pair_pattern.finditer(text):
                try:
                    key = json.loads('"' + match.group(1) + '"')
                    label = json.loads('"' + match.group(2) + '"')
                    parsed[key] = label
                except Exception:
                    continue
    return {canonical_code(k): str(v) for k, v in parsed.items()}


qtext = dict(zip(features["variable"].astype(str), features["question"].astype(str)))
vmaps = {
    str(v): parse_values_json(m)
    for v, m in zip(features["variable"], features["values_json"])
}
question_for = (
    targets.drop_duplicates("question_id")
    .set_index("question_id")["question"]
    .astype(str)
    .to_dict()
)
labels_for = (
    targets.groupby("question_id", sort=False)["label"]
    .apply(lambda x: [str(v) for v in x])
    .to_dict()
)
target_ids = list(labels_for)
target_id_set = set(target_ids)

allowed_features = [
    str(v) for v in features["variable"]
    if str(v) in train.columns and str(v) in test.columns
]
candidate_features = [v for v in allowed_features if v not in target_id_set]

print("Targets:", target_ids)
print("Allowed candidate features:", len(candidate_features))
assert not (set(candidate_features) & target_id_set)


Targets: ['Q201', 'Q73', 'Q227', 'Q209', 'Q33', 'Q148', 'Q17', 'Q186', 'Q242']
Allowed candidate features: 278


## Block 7. Map raw target codes to exact official labels

### Purpose

The leaderboard accepts only exact label strings from the `targets` table. This cell converts numeric target codes in `train` into that official label space.

### What happens

For most targets, the mapping is read directly from the `option` and `label` columns in `targets`.

Two targets require explicit coarsening.

#### Q186

The original 1–10 response scale is grouped into five bins:

- 1–2 → `Never justifiable`;
- 3–4 → `Rarely justifiable`;
- 5–6 → `Sometimes justifiable`;
- 7–8 → `Often justifiable`;
- 9–10 → `Always justifiable`.

#### Q242

The special raw response 0 is kept distinct, while 1–10 is grouped into five ordered levels:

- 0 → `It is against democracy`;
- 1–2 → `Not essential`;
- 3–4 → `Somewhat unimportant`;
- 5–6 → `Moderately essential`;
- 7–8 → `Essential`;
- 9–10 → `Very essential`.

The code verifies that every observed target code has a mapping. It then creates helper functions for decoding one value or an entire target column.

### Main outputs

- `option_to_label`;
- `decode_target_value()`;
- `decode_target_series()`.

### API / cost impact

No API call.

### What to check

The mapping assertions for Q186 and Q242 should pass. Any “unmapped target codes” error means the label-space assumptions do not match the current dataset and inference should not continue.

In [8]:
def build_target_code_maps(train_df: pd.DataFrame, targets_df: pd.DataFrame):
    mappings: dict[str, dict[str, str]] = {}

    for target_id, group in targets_df.groupby("question_id", sort=False):
        target_id = str(target_id)
        ordered_labels = group["label"].astype(str).tolist()
        observed_codes = sort_codes(
            set(train_df[target_id].dropna().map(canonical_code).tolist())
        )

        if target_id == "Q186":
            expected = [
                "Never justifiable", "Rarely justifiable",
                "Sometimes justifiable", "Often justifiable",
                "Always justifiable",
            ]
            assert ordered_labels == expected
            mapping = {
                "1": expected[0], "2": expected[0],
                "3": expected[1], "4": expected[1],
                "5": expected[2], "6": expected[2],
                "7": expected[3], "8": expected[3],
                "9": expected[4], "10": expected[4],
            }
        elif target_id == "Q242":
            expected = [
                "It is against democracy", "Not essential",
                "Somewhat unimportant", "Moderately essential",
                "Essential", "Very essential",
            ]
            assert ordered_labels == expected
            mapping = {
                "0": expected[0],
                "1": expected[1], "2": expected[1],
                "3": expected[2], "4": expected[2],
                "5": expected[3], "6": expected[3],
                "7": expected[4], "8": expected[4],
                "9": expected[5], "10": expected[5],
            }
        else:
            mapping = {
                canonical_code(option): str(label)
                for option, label in zip(group["option"], group["label"])
            }

        missing = set(observed_codes) - set(mapping)
        if missing:
            raise ValueError(f"Unmapped target codes for {target_id}: {sort_codes(missing)}")
        mappings[target_id] = mapping

    return mappings


option_to_label = build_target_code_maps(train, targets)


def decode_target_value(target_id: str, value: Any) -> Optional[str]:
    code = canonical_code(value)
    if code is None:
        return None
    label = option_to_label[target_id].get(code)
    if label is None:
        raise ValueError(f"Unknown code {code!r} for {target_id}")
    return label


def decode_target_series(df: pd.DataFrame, target_id: str) -> pd.Series:
    return df[target_id].map(lambda v: decode_target_value(target_id, v))


for qid in target_ids:
    decoded = decode_target_series(train, qid)
    print(qid, decoded.value_counts(dropna=False).to_dict())


Q201 {'Never': 2007, 'Daily': 900, 'Weekly': 831, 'Less than monthly': 728, 'Monthly': 495, None: 39}
Q73 {'Not very much': 1733, 'None at all': 1527, 'Quite a lot': 1173, 'A great deal': 380, None: 187}
Q227 {'Fairly often': 1237, 'Not often': 1151, 'Very often': 1095, 'Not at all often': 992, None: 525}
Q209 {'Would never do': 2216, 'Might do': 1513, 'Have done': 1121, None: 150}
Q33 {'Disagree': 1566, 'Agree': 1098, 'Agree strongly': 866, 'Neither agree nor disagree': 797, 'Disagree strongly': 636, None: 37}
Q148 {'Very much': 1666, 'Not at all': 1003, 'A great deal': 1000, 'Not much': 951, None: 380}
Q17 {'Not so important': 3305, 'Important': 1616, None: 79}
Q186 {'Never justifiable': 1877, 'Always justifiable': 932, 'Sometimes justifiable': 883, 'Often justifiable': 496, 'Rarely justifiable': 476, None: 336}
Q242 {'Not essential': 1692, 'Moderately essential': 1097, 'Somewhat unimportant': 706, 'Very essential': 582, 'Essential': 569, None: 262, 'It is against democracy': 92}


## Block 8. Create leakage-safe in-domain and out-of-domain splits

### Purpose

This cell defines local validation designs that mimic the two leaderboard settings while keeping validation answers completely separate from model fitting.

### What happens

#### In-domain split

For each country, the function samples up to 30 respondents for validation while retaining at least 20 respondents from that country in the fit set.

This imitates:

- known countries;
- new respondents.

#### Out-of-domain split

The function randomly selects 15 whole countries and removes them entirely from the fit set. Up to 30 respondents from each held-out country are then used for validation.

This imitates:

- unseen countries;
- population shift.

#### Leakage checks

The code asserts that:

- respondent IDs never overlap between fit and validation;
- OOD validation countries do not appear in OOD fit;
- in-domain validation countries remain represented in fit.

#### Target masking

`mask_target_columns()` replaces all target answers with `NaN` before a respondent is passed to the prediction pipeline.

### Main outputs

- `ChallengeSplit`;
- split-construction functions;
- `in_split`;
- `ood_split`;
- `mask_target_columns()`.

### API / cost impact

No LLM requests.

### What to check

The printed OOD held-out country list should be deterministic under seed 42. This exact reproducibility is important because the saved pilot archive was generated from the same split.

In [9]:
@dataclass
class ChallengeSplit:
    fit: pd.DataFrame
    valid: pd.DataFrame
    mode: str
    held_out_countries: list[str] = field(default_factory=list)
    seed: int = 42


def make_in_domain_split(
    df: pd.DataFrame,
    val_per_country: int = 30,
    seed: int = 42,
    min_fit_per_country: int = 20,
) -> ChallengeSplit:
    rng = np.random.default_rng(seed)
    valid_indices = []

    for country, group in df.groupby("country", sort=True):
        max_val = max(1, len(group) - min_fit_per_country)
        n_val = min(val_per_country, max_val)
        chosen = rng.choice(group.index.to_numpy(), size=n_val, replace=False)
        valid_indices.extend(chosen.tolist())

    valid_indices = sorted(valid_indices)
    valid = df.loc[valid_indices].copy().reset_index(drop=True)
    fit = df.drop(index=valid_indices).copy().reset_index(drop=True)

    assert set(valid["country"]).issubset(set(fit["country"]))
    assert set(valid["respondent_id"]).isdisjoint(set(fit["respondent_id"]))
    return ChallengeSplit(fit=fit, valid=valid, mode="in_domain", seed=seed)


def make_out_domain_split(
    df: pd.DataFrame,
    n_countries: int = 15,
    val_per_country: int = 30,
    seed: int = 42,
) -> ChallengeSplit:
    rng = np.random.default_rng(seed)
    countries = np.array(sorted(df["country"].dropna().unique()))
    if n_countries >= len(countries):
        raise ValueError("n_countries must be smaller than total train countries")

    held_out = sorted(rng.choice(countries, size=n_countries, replace=False).tolist())
    fit = df.loc[~df["country"].isin(held_out)].copy().reset_index(drop=True)

    valid_parts = []
    held_pool = df.loc[df["country"].isin(held_out)]
    for country, group in held_pool.groupby("country", sort=True):
        n = min(val_per_country, len(group))
        chosen = rng.choice(group.index.to_numpy(), size=n, replace=False)
        valid_parts.append(df.loc[chosen])

    valid = pd.concat(valid_parts, ignore_index=True)
    assert set(valid["country"]).isdisjoint(set(fit["country"]))
    assert set(valid["respondent_id"]).isdisjoint(set(fit["respondent_id"]))
    return ChallengeSplit(
        fit=fit,
        valid=valid,
        mode="out_of_domain",
        held_out_countries=held_out,
        seed=seed,
    )


def mask_target_columns(df: pd.DataFrame) -> pd.DataFrame:
    masked = df.copy()
    for target_id in target_ids:
        if target_id in masked.columns:
            masked[target_id] = np.nan
    return masked


in_split = make_in_domain_split(
    train,
    val_per_country=CFG.in_domain_val_per_country,
    seed=CFG.seed,
)
ood_split = make_out_domain_split(
    train,
    n_countries=CFG.out_domain_n_countries,
    val_per_country=CFG.out_domain_val_per_country,
    seed=CFG.seed,
)

print("IN-DOMAIN:", in_split.fit.shape, in_split.valid.shape)
print("OOD:", ood_split.fit.shape, ood_split.valid.shape)
print("OOD held-out countries:", ood_split.held_out_countries)


IN-DOMAIN: (3500, 345) (1500, 345)
OOD: (3500, 345) (450, 345)
OOD held-out countries: ['Bolivia', 'Brazil', 'Cyprus', 'Indonesia', 'Libya', 'Mongolia', 'Morocco', 'Peru', 'Romania', 'Serbia', 'Slovakia', 'Ukraine', 'United States', 'Uruguay', 'Vietnam']


## Block 9. Select target-specific features with NMI and mRMR

### Purpose

Different targets require different respondent evidence. This cell builds a separate feature ranking for each target using only the fit data.

### What happens

#### Step 1: encode categorical answers

Every allowed feature and every decoded target is factorised into integer category codes. Missing values receive code `-1`.

This makes repeated NMI calculations much faster.

#### Step 2: measure target relevance

For every target–feature pair, the selector calculates normalized mutual information:

\[
NMI(X_j,Y_t).
\]

NMI is appropriate because the survey variables are categorical and relationships need not be linear.

A feature is eligible only when:

- it is observed for at least 25% of rows where the target is observed;
- it has at least 100 paired observations.

The relevance score is adjusted by:

\[
NMI \times \sqrt{\text{availability}} \times \text{geographic penalty}.
\]

`N_REGION_ISO` is penalised in OOD and mixed modes to reduce overdependence on geographic identifiers.

#### Step 3: reduce redundancy with mRMR

The selector starts from the 40 highest-ranked features and chooses up to 18.

For each candidate:

\[
\text{selection score}
=
\text{target relevance}
-
0.20 \times \text{maximum redundancy with selected features}.
\]

This prevents the prompt from being filled with several near-duplicate questions.

#### Step 4: normalise weights

The selected relevance scores are scaled to the 0–1 range for use in retrieval and hybrid importance.

### Main outputs

`TargetFeatureSelector`, which later stores:

- full rankings per target;
- 18 selected mRMR features per target;
- normalised selector weights.

### API / cost impact

No LLM call. This is statistical preprocessing only.

### What to check

This cell defines the selector but does not fit it yet. Fitting happens in Blocks 20 and 27.

During review, focus on:

- why NMI is used;
- why availability is penalised;
- why mRMR is needed;
- whether geographic penalties are appropriate for the intended version.

In [10]:
def safe_nmi(x: pd.Series, y: pd.Series) -> float:
    mask = x.notna() & y.notna()
    if mask.sum() < 2:
        return 0.0
    x2 = x.loc[mask].map(canonical_code).astype(str)
    y2 = y.loc[mask].astype(str)
    if x2.nunique() < 2 or y2.nunique() < 2:
        return 0.0
    return float(normalized_mutual_info_score(y2, x2))


class TargetFeatureSelector:
    def __init__(self, config: PipelineConfig):
        self.config = config
        self.rankings: dict[str, pd.DataFrame] = {}
        self.selected: dict[str, list[str]] = {}
        self.weights: dict[str, dict[str, float]] = {}
        self.fit_df: Optional[pd.DataFrame] = None
        self.mode: str = "mixed"
        self.feature_codes: dict[str, np.ndarray] = {}
        self.target_codes: dict[str, np.ndarray] = {}

    def _geo_penalty(self, variable: str) -> float:
        if variable != "N_REGION_ISO":
            return 1.0
        if self.mode == "out_of_domain":
            return self.config.ood_region_penalty
        if self.mode == "mixed":
            return self.config.mixed_region_penalty
        return 1.0

    @staticmethod
    def _factorize(series: pd.Series) -> np.ndarray:
        canonical = series.map(canonical_code)
        codes, _ = pd.factorize(canonical, sort=False)
        return codes.astype(np.int32, copy=False)  # missing values are -1

    @staticmethod
    def _nmi_from_codes(left: np.ndarray, right: np.ndarray, mask: np.ndarray) -> float:
        if int(mask.sum()) < 2:
            return 0.0
        x = left[mask]
        y = right[mask]
        if np.unique(x).size < 2 or np.unique(y).size < 2:
            return 0.0
        return float(normalized_mutual_info_score(y, x))

    def _prepare_codes(self):
        assert self.fit_df is not None
        self.feature_codes = {
            variable: self._factorize(self.fit_df[variable])
            for variable in tqdm(candidate_features, desc="Encoding feature columns")
        }
        self.target_codes = {}
        for target_id in target_ids:
            y = decode_target_series(self.fit_df, target_id)
            codes, _ = pd.factorize(y, sort=False)
            self.target_codes[target_id] = codes.astype(np.int32, copy=False)

    def _rank_one(self, target_id: str) -> pd.DataFrame:
        y_codes = self.target_codes[target_id]
        target_observed = y_codes >= 0
        target_rows = int(target_observed.sum())
        records = []

        for variable in candidate_features:
            x_codes = self.feature_codes[variable]
            paired = target_observed & (x_codes >= 0)
            n = int(paired.sum())
            if target_rows == 0:
                continue
            availability = n / target_rows
            if availability < self.config.min_availability or n < self.config.min_paired_rows:
                continue

            nmi = self._nmi_from_codes(x_codes, y_codes, paired)
            adjusted = nmi * math.sqrt(availability) * self._geo_penalty(variable)
            records.append({
                "feature": variable,
                "question": qtext.get(variable, variable),
                "nmi": nmi,
                "availability": availability,
                "paired_rows": n,
                "adjusted_score": adjusted,
            })

        if not records:
            raise ValueError(f"No eligible features for {target_id}")
        return (
            pd.DataFrame(records)
            .sort_values(["adjusted_score", "availability"], ascending=False)
            .reset_index(drop=True)
        )

    def _redundancy(self, left: str, right: str) -> float:
        x = self.feature_codes[left]
        y = self.feature_codes[right]
        mask = (x >= 0) & (y >= 0)
        if int(mask.sum()) < self.config.min_paired_rows:
            return 0.0
        return self._nmi_from_codes(x, y, mask)

    def _mrmr_select(self, ranking: pd.DataFrame) -> list[str]:
        pool = ranking.head(self.config.selection_pool_size).copy()
        relevance = dict(zip(pool["feature"], pool["adjusted_score"]))
        remaining = list(pool["feature"])
        selected: list[str] = []
        pair_cache: dict[tuple[str, str], float] = {}

        while remaining and len(selected) < self.config.selected_features:
            best_feature = None
            best_score = -np.inf

            for feature in remaining:
                if not selected:
                    redundancy = 0.0
                else:
                    vals = []
                    for chosen in selected:
                        key = tuple(sorted((feature, chosen)))
                        if key not in pair_cache:
                            pair_cache[key] = self._redundancy(feature, chosen)
                        vals.append(pair_cache[key])
                    redundancy = max(vals) if vals else 0.0

                score = relevance[feature] - self.config.redundancy_lambda * redundancy
                if score > best_score:
                    best_feature = feature
                    best_score = score

            selected.append(best_feature)
            remaining.remove(best_feature)

        return selected

    def fit(self, fit_df: pd.DataFrame, mode: str):
        self.fit_df = fit_df.reset_index(drop=True)
        self.mode = mode
        self.rankings = {}
        self.selected = {}
        self.weights = {}
        self._prepare_codes()

        for target_id in tqdm(target_ids, desc="Target feature selection"):
            ranking = self._rank_one(target_id)
            selected = self._mrmr_select(ranking)
            self.rankings[target_id] = ranking
            self.selected[target_id] = selected

            raw_weights = (
                ranking.set_index("feature")["adjusted_score"]
                .reindex(selected)
                .fillna(0.0)
            )
            scale = float(raw_weights.max()) or 1.0
            self.weights[target_id] = (raw_weights / scale).to_dict()
        return self

    def available_prompt_features(self, row: pd.Series, target_id: str) -> list[str]:
        ordered = self.rankings[target_id]["feature"].tolist()
        available = [v for v in ordered if not pd.isna(row.get(v))]
        return available[: self.config.prompt_features]


## Block 10. Fit-only priors and the CatBoost auxiliary expert

### Purpose

This cell defines the statistical expert that supports the LLM without replacing it.

### What happens

#### Majority labels and class priors

`compute_fit_priors()` calculates, separately for each target:

- the majority label;
- the observed training distribution across official labels.

These values are calculated only from the fit fold, which avoids validation leakage.

#### Categorical preparation

All selected feature values are converted to stable strings. Missing values become the explicit category:

```text
__MISSING__
```

This lets CatBoost treat missingness as potentially informative.

#### One CatBoost model per target

`CatBoostExpert` trains nine independent classifiers.

The model uses:

- balanced class weights, helping rare labels;
- categorical predictors;
- 180 trees;
- depth 6;
- learning rate 0.08;
- deterministic seed.

It provides:

1. class probabilities for each respondent;
2. a hard prediction;
3. feature importance;
4. a fallback prediction if LLM output is unusable.

### Main outputs

- `compute_fit_priors()`;
- `categorical_frame()`;
- `CatBoostExpert`.

### API / cost impact

No LLM cost. CatBoost training uses CPU and usually completes quickly relative to inference.

### What to check

This cell only defines the expert. The models are fitted later.

Important conceptual point: CatBoost probabilities are auxiliary. In normal valid cases, they receive only 12% of the final probability weight.

In [11]:
def compute_fit_priors(fit_df: pd.DataFrame):
    majority = {}
    priors = {}
    for target_id in target_ids:
        y = decode_target_series(fit_df, target_id).dropna()
        shares = y.value_counts(normalize=True).reindex(labels_for[target_id], fill_value=0.0)
        majority[target_id] = str(shares.idxmax())
        priors[target_id] = shares.to_dict()
    return majority, priors


MISSING_TOKEN = "__MISSING__"


def categorical_frame(df: pd.DataFrame, variables: list[str]) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for variable in variables:
        out[variable] = df[variable].map(canonical_code).fillna(MISSING_TOKEN).astype(str)
    return out


class CatBoostExpert:
    def __init__(self, config: PipelineConfig):
        self.config = config
        self.models = {}
        self.features = {}
        self.constants = {}
        self.available = False

    def fit(self, fit_df: pd.DataFrame, selected_features: dict[str, list[str]]):
        if not self.config.use_catboost:
            return self
        try:
            from catboost import CatBoostClassifier
        except Exception as exc:
            print("CatBoost unavailable; continuing without it:", exc)
            return self

        for target_id in target_ids:
            y = decode_target_series(fit_df, target_id)
            mask = y.notna()
            y_fit = y.loc[mask].astype(str)
            vars_ = selected_features[target_id][: self.config.retrieval_features]
            self.features[target_id] = vars_

            if y_fit.nunique() < 2:
                self.constants[target_id] = str(y_fit.mode().iloc[0])
                continue

            X = categorical_frame(fit_df.loc[mask], vars_)
            loss = "Logloss" if y_fit.nunique() == 2 else "MultiClass"
            model = CatBoostClassifier(
                iterations=self.config.catboost_iterations,
                depth=self.config.catboost_depth,
                learning_rate=self.config.catboost_learning_rate,
                loss_function=loss,
                auto_class_weights="Balanced",
                random_seed=self.config.seed,
                verbose=False,
                allow_writing_files=False,
                thread_count=-1,
            )
            model.fit(X, y_fit, cat_features=list(range(X.shape[1])))
            self.models[target_id] = model

        self.available = bool(self.models or self.constants)
        return self

    def predict_proba(self, row: pd.Series, target_id: str) -> Optional[dict[str, float]]:
        labels = labels_for[target_id]
        if target_id in self.constants:
            result = {label: 0.0 for label in labels}
            result[self.constants[target_id]] = 1.0
            return result
        model = self.models.get(target_id)
        if model is None:
            return None

        vars_ = self.features[target_id]
        X = categorical_frame(pd.DataFrame([row]), vars_)
        probs = model.predict_proba(X)[0]
        result = {label: 0.0 for label in labels}
        for cls, prob in zip(model.classes_, probs):
            if str(cls) in result:
                result[str(cls)] = float(prob)
        total = sum(result.values())
        return {k: v / total for k, v in result.items()} if total > 0 else None


## Block 11. Retrieve similar labelled respondents

### Purpose

This cell defines the compact retrieval-augmented generation component. The LLM sees a small number of real training respondents who resemble the current respondent on target-relevant features.

### What happens

For each target, a separate retrieval index is created from fit respondents with a known target answer.

#### Similarity calculation

The retriever compares the new respondent with fit respondents on the target-specific feature list.

Features contribute more when their selector/hybrid weight is larger. Missing comparisons are ignored rather than treated as mismatches.

The score rewards:

- exact categorical matches;
- coverage across more comparable features.

#### Example selection

The retriever chooses up to two examples.

It can prefer one same-country example in in-domain mode, then fill remaining slots globally. It also limits the number of examples from any one target label to reduce majority-class copying.

#### What is displayed later

Each retrieved example contributes:

- a similarity score;
- three relevant survey answers;
- its observed target label.

### Main outputs

`SimilarRespondentRetriever`.

A separate fitted instance is created for every target in the main pipeline.

### API / cost impact

No API call. Retrieval is local computation.

### What to check

The retrieved target labels come only from the fit data. The current respondent's hidden target answer is never used.

During prompt preview, verify that retrieved examples are plausible analogues but not treated as deterministic rules.

In [12]:
class SimilarRespondentRetriever:
    def __init__(
        self,
        fit_df: pd.DataFrame,
        target_id: str,
        variables: list[str],
        weights: dict[str, float],
        config: PipelineConfig,
    ):
        self.target_id = target_id
        self.variables = variables
        self.config = config

        y = decode_target_series(fit_df, target_id)
        mask = y.notna()
        self.df = fit_df.loc[mask].copy().reset_index(drop=True)
        self.labels = y.loc[mask].astype(str).reset_index(drop=True)
        self.countries = self.df["country"].astype(str).to_numpy()
        self.respondent_ids = self.df["respondent_id"].astype(str).to_numpy()

        X = categorical_frame(self.df, variables)
        self.matrix = X.to_numpy(dtype=object)
        self.weights = np.array([max(float(weights.get(v, 0.05)), 0.02) for v in variables])

    def _scores(self, row: pd.Series) -> np.ndarray:
        q = np.array([
            canonical_code(row.get(v)) if not pd.isna(row.get(v)) else MISSING_TOKEN
            for v in self.variables
        ], dtype=object)
        q[q == None] = MISSING_TOKEN  # noqa: E711

        valid = (self.matrix != MISSING_TOKEN) & (q[None, :] != MISSING_TOKEN)
        denom = (valid * self.weights[None, :]).sum(axis=1)
        matches = (self.matrix == q[None, :]) & valid
        numer = (matches * self.weights[None, :]).sum(axis=1)
        coverage = valid.mean(axis=1)
        score = np.divide(numer, denom, out=np.zeros_like(numer, dtype=float), where=denom > 0)
        return score + 0.03 * coverage

    def _choose_diverse(self, indices: np.ndarray, scores: np.ndarray, n: int, banned: set[int]):
        chosen = []
        label_counts = Counter()
        ordered = indices[np.argsort(scores[indices])[::-1]]
        for idx in ordered:
            idx = int(idx)
            if idx in banned:
                continue
            label = self.labels.iloc[idx]
            if label_counts[label] >= self.config.max_examples_per_label:
                continue
            chosen.append(idx)
            label_counts[label] += 1
            if len(chosen) >= n:
                break
        return chosen

    def retrieve(self, row: pd.Series, row_mode: str) -> list[dict[str, Any]]:
        if not self.config.use_retrieval:
            return []
        scores = self._scores(row)
        country = str(row.get("country", ""))
        all_idx = np.arange(len(self.df))
        chosen: list[int] = []

        if row_mode == "in_domain" and self.config.n_same_country_examples > 0:
            same_idx = all_idx[self.countries == country]
            chosen.extend(self._choose_diverse(
                same_idx,
                scores,
                min(self.config.n_same_country_examples, self.config.n_retrieval_examples),
                banned=set(),
            ))

        remaining_n = self.config.n_retrieval_examples - len(chosen)
        if remaining_n > 0:
            global_idx = all_idx
            chosen.extend(self._choose_diverse(
                global_idx,
                scores,
                remaining_n,
                banned=set(chosen),
            ))

        examples = []
        for idx in chosen:
            examples.append({
                "respondent": self.df.iloc[idx],
                "target_label": self.labels.iloc[idx],
                "similarity": float(scores[idx]),
                "country": self.countries[idx],
                "respondent_id": self.respondent_ids[idx],
            })
        return examples


## Block 12. Define target-specific semantic guides

### Purpose

The selected survey answers are statistical signals, but the LLM still needs a concise explanation of the latent concept behind each target. This cell provides one short reasoning guide per target.

### What happens

`TARGET_GUIDES` describes the intended construct, for example:

- institutional trust;
- electoral integrity;
- non-electoral political participation;
- gender-role traditionalism;
- security anxiety;
- obedience versus autonomy;
- moral permissiveness;
- religious authority in democratic law.

These guides do **not** choose features. Feature selection remains data-driven through NMI, mRMR and CatBoost importance.

`LABEL_NOTES` explains the special official bins for Q186 and Q242.

`PROFILE_DIMENSIONS` lists broad latent dimensions used in the earlier full alternative pipeline. Fast9 does not run a separate profile-generation stage, so this list is retained mainly for compatibility/documentation and does not drive normal Fast9 inference.

### Main outputs

- `TARGET_GUIDES`;
- `LABEL_NOTES`;
- `PROFILE_DIMENSIONS`.

### API / cost impact

No API call.

### What to check

The guides should clarify the target without imposing a rigid stereotype. They are semantic priors, not causal rules.

Changing a guide can change LLM predictions and therefore creates a new model version requiring validation.

In [13]:
TARGET_GUIDES = {
    "Q201": (
        "Infer habitual information-source use. Give most weight to the respondent's use of "
        "other news and communication channels, general media engagement, education and age. "
        "Distinguish daily, weekly, occasional and never use rather than merely online/offline preference."
    ),
    "Q73": (
        "Infer latent institutional trust. Give strongest weight to confidence in government, political parties, "
        "elections, courts, civil service and police. Use country only as weak contextual evidence."
    ),
    "Q227": (
        "Infer the respondent's perception of electoral integrity. Prioritise answers about vote buying, threats, "
        "fair counting, media fairness, corruption and political trust."
    ),
    "Q209": (
        "Infer willingness to participate in non-electoral politics. Prioritise past or potential participation in "
        "petitions, boycotts, demonstrations, online activism and civic organisations."
    ),
    "Q33": (
        "Infer gender-role traditionalism concerning employment. Prioritise views on women's leadership, education, "
        "family roles, motherhood and equality. Do not treat country as determinative."
    ),
    "Q148": (
        "Infer perceived personal or national security risk. Prioritise worries about war, terrorism, political conflict, "
        "economic insecurity and neighbourhood safety. Distinguish high concern from a generally negative worldview."
    ),
    "Q17": (
        "Infer preference for child obedience versus autonomy. Consider religion, authority orientation, family values, "
        "traditionalism and whether independence is valued for children."
    ),
    "Q186": (
        "Infer sexual-morality permissiveness. Prioritise views on divorce, homosexuality, abortion, prostitution, "
        "casual sex, religiosity and social conservatism. Map the inferred 1-10 position to the supplied bins."
    ),
    "Q242": (
        "Infer support for religious authority in democratic law. Consider religiosity, church-state preferences, "
        "democratic values, authoritarianism and confidence in religious institutions. Respect the special anti-democratic category."
    ),
}

LABEL_NOTES = {
    "Q186": (
        "Official bins: Never justifiable = raw 1-2; Rarely justifiable = 3-4; "
        "Sometimes justifiable = 5-6; Often justifiable = 7-8; Always justifiable = 9-10."
    ),
    "Q242": (
        "Official bins: It is against democracy = special raw response 0; Not essential = 1-2; "
        "Somewhat unimportant = 3-4; Moderately essential = 5-6; Essential = 7-8; Very essential = 9-10."
    ),
}

PROFILE_DIMENSIONS = [
    "institutional_trust",
    "political_participation",
    "electoral_integrity_concern",
    "gender_traditionalism",
    "security_anxiety",
    "religiosity",
    "authority_orientation",
    "social_conservatism",
    "media_engagement",
]


## Block 13. Cache API responses and parse structured probabilities

### Purpose

This cell makes LLM inference reliable, reproducible and restartable.

### What happens

#### Cached Nebius client

`CachedNebiusClient` wraps the raw API client.

For every request, it creates a deterministic cache key from:

- model;
- system prompt;
- user prompt;
- temperature;
- maximum output length;
- thinking setting.

If an identical request already exists in the JSONL cache, the response is returned locally instead of calling Nebius again.

When a new request is needed, the client:

- sends the request;
- retries transient failures up to the configured limit;
- waits between attempts;
- stores successful output in the cache.

A lock protects cache writes when many worker threads finish simultaneously.

#### Structured response parsing

The remaining helper functions:

- normalise label spelling;
- extract a JSON object even if extra text surrounds it;
- validate that labels belong to the official label set;
- coerce probability values to numbers;
- clip invalid negative values;
- renormalise probabilities to sum to one;
- calculate the margin between the two highest probabilities.

### Main outputs

- `CachedNebiusClient`;
- response-parsing helpers;
- probability-margin helper.

### API / cost impact

Defining the class costs nothing. Paid calls occur only when its request method is used later and the prompt is not already cached.

### What to check

Cache paths must be different for genuinely different model versions. Otherwise an old response could be reused for a modified prompt.

The parser is deliberately strict about official labels because invalid strings would be punished by the challenge evaluation.

In [14]:
class CachedNebiusClient:
    def __init__(
        self,
        config: PipelineConfig,
        api_key: Optional[str] = None,
        client: Optional[Any] = None,
    ):
        self.config = config
        if OpenAI is None:
            raise ImportError("Install the openai package first: pip install openai")

        if client is not None:
            self.client = client
        else:
            api_key = api_key or os.getenv("NEBIUS_API_KEY")
            if not api_key:
                api_key = getpass("Nebius API key: ").strip()
            if not api_key:
                raise ValueError("Nebius API key is empty")
            os.environ["NEBIUS_API_KEY"] = api_key
            self.client = OpenAI(base_url=config.base_url, api_key=api_key)
        self.cache_path = Path(config.cache_path)
        self.lock = threading.Lock()
        self.cache: dict[str, str] = {}
        if self.cache_path.exists():
            with self.cache_path.open("r", encoding="utf-8") as f:
                for line in f:
                    try:
                        record = json.loads(line)
                        self.cache[record["key"]] = record["response"]
                    except Exception:
                        pass
        print(f"Loaded {len(self.cache)} cached LLM responses")

    def _key(self, system: str, user: str, thinking: bool, max_tokens: int) -> str:
        payload = json.dumps({
            "model": self.config.model,
            "system": system,
            "user": user,
            "thinking": thinking,
            "max_tokens": max_tokens,
            "temperature": self.config.temperature,
        }, ensure_ascii=False, sort_keys=True)
        return hashlib.sha256(payload.encode("utf-8")).hexdigest()

    def complete(self, system: str, user: str, thinking: bool = False, max_tokens: int = 320) -> str:
        key = self._key(system, user, thinking, max_tokens)
        with self.lock:
            cached = self.cache.get(key)
        if cached is not None:
            return cached

        last_error = None
        for attempt in range(self.config.retries):
            try:
                response = self.client.chat.completions.create(
                    model=self.config.model,
                    messages=[
                        {"role": "system", "content": system},
                        {"role": "user", "content": user},
                    ],
                    temperature=self.config.temperature,
                    max_tokens=max_tokens,
                    extra_body={"chat_template_kwargs": {"enable_thinking": thinking}},
                )
                text = response.choices[0].message.content or ""
                with self.lock:
                    if key not in self.cache:
                        self.cache[key] = text
                        with self.cache_path.open("a", encoding="utf-8") as f:
                            f.write(json.dumps({"key": key, "response": text}, ensure_ascii=False) + "\n")
                return text
            except Exception as exc:
                last_error = exc
                if attempt < self.config.retries - 1:
                    time.sleep(min(2 ** attempt, 10) + random.random())
        raise last_error


def normalise_label(text: Any, labels: list[str]) -> Optional[str]:
    cleaned = re.sub(r"\s+", " ", str(text).strip().strip('"').strip("'"))
    for label in labels:
        if cleaned == label or cleaned.casefold() == label.casefold():
            return label
    matches = [label for label in labels if label.casefold() in cleaned.casefold()]
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        matches = sorted(matches, key=len, reverse=True)
        if len(matches[0]) > len(matches[1]):
            return matches[0]
    return None


def extract_json_object(text: str) -> Optional[dict[str, Any]]:
    cleaned = re.sub(r"```(?:json)?", "", str(text), flags=re.I).replace("```", "").strip()
    try:
        obj = json.loads(cleaned)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start >= 0 and end > start:
        try:
            obj = json.loads(cleaned[start:end + 1])
            return obj if isinstance(obj, dict) else None
        except Exception:
            return None
    return None


def parse_probability_response(text: str, labels: list[str]):
    obj = extract_json_object(text)
    prediction = None
    probs = {label: 0.0 for label in labels}

    if obj is not None:
        prediction = normalise_label(obj.get("prediction"), labels)
        raw_probs = obj.get("probabilities", {})
        if isinstance(raw_probs, dict):
            for raw_label, raw_prob in raw_probs.items():
                label = normalise_label(raw_label, labels)
                if label is None:
                    continue
                try:
                    probs[label] = max(float(raw_prob), 0.0)
                except Exception:
                    pass

    if prediction is None:
        prediction = normalise_label(text, labels)

    total = sum(probs.values())
    if total > 0:
        probs = {k: v / total for k, v in probs.items()}
    elif prediction is not None:
        probs[prediction] = 1.0
    else:
        return None

    if prediction is None:
        prediction = max(probs, key=probs.get)
    return {"prediction": prediction, "probabilities": probs}


def probability_margin(probs: dict[str, float]) -> float:
    values = sorted(probs.values(), reverse=True)
    return values[0] - values[1] if len(values) > 1 else 1.0


## Block 14. Render respondent evidence as readable text

### Purpose

The raw survey tables contain numeric answer codes. This cell converts selected respondent features into concise natural-language evidence suitable for an LLM prompt.

### What happens

`answer_text()`:

1. canonicalises a raw answer code;
2. looks up its text label in the official feature mapping;
3. returns `None` for missing or non-substantive values;
4. falls back to the code itself if no text mapping exists.

`render_evidence()` then formats a list of selected features as readable lines containing:

- variable code;
- question text;
- decoded answer.

The function respects a maximum number of features and skips missing answers.

### Main outputs

- `answer_text()`;
- `render_evidence()`.

### API / cost impact

No API call.

### What to check

Prompt previews should show understandable answer text rather than unexplained numeric codes. If many values remain numeric, inspect the corresponding `values_json` metadata.

In [15]:
def answer_text(variable: str, value: Any) -> Optional[str]:
    code = canonical_code(value)
    if code is None:
        return None
    mapping = vmaps.get(variable, {})
    if variable == "N_REGION_ISO" and code not in mapping:
        return None
    return mapping.get(code, code)


def render_evidence(
    row: pd.Series,
    variables: list[str],
    max_items: Optional[int] = None,
) -> str:
    blocks = []
    for variable in variables:
        value = row.get(variable)
        if pd.isna(value):
            continue
        decoded = answer_text(variable, value)
        if decoded is None:
            continue
        blocks.append(
            f"- [{variable}] {qtext.get(variable, variable)}\n"
            f"  Answer: {decoded}"
        )
        if max_items is not None and len(blocks) >= max_items:
            break
    return "\n".join(blocks) if blocks else "No usable answers available."


## Block 15. Combine selector relevance and CatBoost importance

### Purpose

NMI/mRMR captures strong one-feature associations and diversity, while CatBoost can identify nonlinear and interaction-based importance. This cell combines both sources conservatively.

### What happens

#### CatBoost importance

For each target, the code extracts feature importance from the fitted CatBoost model and normalises it.

#### Protected mRMR core

The first eight mRMR-selected features are protected. This prevents the final feature set from becoming entirely driven by CatBoost.

#### CatBoost additions

Up to four highly important CatBoost features are added when they are not already protected.

#### Hybrid score

The remaining slots are filled using:

\[
0.70 \times \text{selector importance}
+
0.20 \times \text{CatBoost importance}
+
0.10 \times \text{availability}.
\]

The final list contains up to 16 features per target.

From that list:

- up to 12 observed answers enter the direct prompt;
- up to 16 features are available for similarity retrieval.

### Main outputs

- `catboost_feature_importance()`;
- `build_hybrid_feature_plan()`.

The fitted pipeline later stores the resulting ranking and an audit table.

### API / cost impact

No LLM call.

### What to check

This cell defines the combination logic but does not fit anything yet.

During team review, the central design decision is the 70/20/10 weighting and the protection of the first eight mRMR features.

In [16]:
def catboost_feature_importance(
    expert: CatBoostExpert,
    target_id: str,
) -> dict[str, float]:
    model = expert.models.get(target_id)
    variables = expert.features.get(target_id, [])

    if model is None or not variables:
        return {variable: 0.0 for variable in variables}

    raw = model.get_feature_importance()
    result = {
        variable: max(float(value), 0.0)
        for variable, value in zip(variables, raw)
    }
    scale = max(result.values(), default=0.0)

    if scale > 0:
        result = {
            variable: value / scale
            for variable, value in result.items()
        }
    return result


def build_hybrid_feature_plan(
    selector: TargetFeatureSelector,
    expert: CatBoostExpert,
    config: FastNineCallConfig,
):
    hybrid_rankings = {}
    hybrid_selected = {}
    hybrid_weights = {}
    importance_rows = []

    for target_id in target_ids:
        ranking = selector.rankings[target_id].copy()

        selector_max = float(
            ranking["adjusted_score"].max()
        ) or 1.0
        ranking["selector_norm"] = (
            ranking["adjusted_score"] / selector_max
        )

        cb_map = catboost_feature_importance(
            expert,
            target_id,
        )
        ranking["catboost_importance"] = (
            ranking["feature"]
            .map(cb_map)
            .fillna(0.0)
        )

        ranking["hybrid_score"] = (
            config.selector_importance_weight
            * ranking["selector_norm"]
            + config.catboost_importance_weight
            * ranking["catboost_importance"]
            + config.availability_weight
            * ranking["availability"].clip(0.0, 1.0)
        )

        ranking = ranking.sort_values(
            ["hybrid_score", "adjusted_score", "availability"],
            ascending=False,
        ).reset_index(drop=True)

        protected = list(
            selector.selected[target_id][
                : config.protected_mrmr_features
            ]
        )

        cb_top = [
            feature
            for feature, value in sorted(
                cb_map.items(),
                key=lambda pair: pair[1],
                reverse=True,
            )
            if value > 0
        ][: config.catboost_importance_features]

        selected = []
        for feature in protected + cb_top + ranking["feature"].tolist():
            if feature not in selected:
                selected.append(feature)
            if len(selected) >= config.hybrid_selected_features:
                break

        hybrid_rankings[target_id] = ranking
        hybrid_selected[target_id] = selected

        scores = (
            ranking.set_index("feature")["hybrid_score"]
            .reindex(selected)
            .fillna(0.0)
        )
        score_scale = float(scores.max()) or 1.0
        hybrid_weights[target_id] = (
            scores / score_scale
        ).to_dict()

        selected_set = set(selected)
        protected_set = set(protected)
        cb_top_set = set(cb_top)

        for _, item in ranking.iterrows():
            importance_rows.append({
                "question_id": target_id,
                "feature": item["feature"],
                "question": item["question"],
                "selected_for_prompt_or_retrieval": (
                    item["feature"] in selected_set
                ),
                "protected_mrmr": (
                    item["feature"] in protected_set
                ),
                "catboost_top_addition": (
                    item["feature"] in cb_top_set
                ),
                "nmi": float(item["nmi"]),
                "availability": float(item["availability"]),
                "adjusted_score": float(item["adjusted_score"]),
                "selector_norm": float(item["selector_norm"]),
                "catboost_importance": float(
                    item["catboost_importance"]
                ),
                "hybrid_score": float(item["hybrid_score"]),
            })

    return (
        hybrid_rankings,
        hybrid_selected,
        hybrid_weights,
        pd.DataFrame(importance_rows),
    )


## Block 16. Define the complete Fast9 LLM-primary pipeline

### Purpose

This is the central modelling cell. It connects feature selection, CatBoost, retrieval, prompt construction, LLM inference, parsing, repair and final probability blending.

### What happens during `.fit()`

For each target, the pipeline:

1. decodes fit target labels;
2. fits the NMI/mRMR selector;
3. calculates fit-only majority labels and class priors;
4. trains the target-specific CatBoost model;
5. builds the hybrid feature plan;
6. fits the target-specific retrieval index;
7. stores seen countries and diagnostic feature tables.

No LLM calls occur during fitting.

### What happens during one prediction

For one respondent and one target:

1. all hidden target columns are unavailable/masked;
2. the pipeline selects up to 12 observed target-relevant answers;
3. it retrieves up to two labelled fit respondents;
4. CatBoost produces a probability prior;
5. a target-specific prompt is assembled;
6. Qwen returns one probability for every official label;
7. the output is parsed and validated;
8. if invalid, one short repair call is attempted;
9. if still invalid, CatBoost becomes the technical fallback;
10. valid LLM and CatBoost probabilities are blended:

\[
P_{\text{final}}
=
0.88P_{\text{LLM}}
+
0.12P_{\text{CatBoost}}.
\]

The label with the largest final probability becomes the prediction.

### System prompts

`PREDICTION_SYSTEM` instructs the LLM to:

- predict the respondent rather than express its own opinion;
- prioritise direct respondent evidence;
- treat retrieval and CatBoost as supporting evidence;
- return valid JSON with probabilities for every label.

`REPAIR_SYSTEM` asks the model only to repair malformed structure, not to rethink the substantive answer.

### Main outputs

- `FastNineCallLLMPipeline`;
- system prompts used for normal and repair calls.

### API / cost impact

Defining the class costs nothing. API requests occur only when `.predict_one()` is called through the parallel runner.

### What to check

This is the most important code-review block. The team should confirm:

- no hidden target answer reaches the prompt;
- labels are exact;
- blending weights are correct;
- repair is format-only;
- fallback guarantees 100% coverage;
- prompt content matches the intended methodology.

In [17]:
PREDICTION_SYSTEM = """
You are a calibrated survey-response prediction model.
Predict how this specific real respondent answered the target survey question.
This is a classification task, not a request for your own opinion.
Use the respondent's direct answers first, retrieved respondents only as analogies,
and country only as weak context. The CatBoost prior is auxiliary evidence and
must not replace respondent-specific reasoning.
Return valid JSON only and assign probabilities to every allowed label.
""".strip()


REPAIR_SYSTEM = """
You repair structured classification output.
Return valid JSON only. Do not add explanation or markdown.
""".strip()


class FastNineCallLLMPipeline:
    def __init__(
        self,
        config: FastNineCallConfig,
        llm: CachedNebiusClient,
    ):
        self.config = config
        self.llm = llm
        self.fit_df = None
        self.mode = "mixed"
        self.seen_countries = set()

        self.selector = TargetFeatureSelector(config)
        self.majority = {}
        self.priors = {}
        self.expert = CatBoostExpert(config)

        self.hybrid_rankings = {}
        self.hybrid_selected = {}
        self.hybrid_weights = {}
        self.feature_importance_table = pd.DataFrame()

        self.retrievers = {}

    def fit(
        self,
        fit_df: pd.DataFrame,
        mode: str = "mixed",
    ):
        self.fit_df = fit_df.copy().reset_index(drop=True)
        self.mode = mode
        self.seen_countries = set(
            self.fit_df["country"].astype(str)
        )

        started = time.perf_counter()

        self.selector.fit(
            self.fit_df,
            mode=mode,
        )
        self.majority, self.priors = compute_fit_priors(
            self.fit_df
        )

        catboost_pool = {}
        for target_id in target_ids:
            pool = (
                self.selector.selected[target_id]
                + self.selector.rankings[target_id]
                    .head(self.config.catboost_pool_features)[
                        "feature"
                    ]
                    .tolist()
            )
            catboost_pool[target_id] = list(
                dict.fromkeys(pool)
            )[: self.config.catboost_pool_features]

        self.expert.fit(
            self.fit_df,
            catboost_pool,
        )

        (
            self.hybrid_rankings,
            self.hybrid_selected,
            self.hybrid_weights,
            self.feature_importance_table,
        ) = build_hybrid_feature_plan(
            self.selector,
            self.expert,
            self.config,
        )

        self.retrievers = {}
        for target_id in target_ids:
            variables = self.hybrid_selected[target_id][
                : self.config.retrieval_features
            ]
            self.retrievers[target_id] = (
                SimilarRespondentRetriever(
                    fit_df=self.fit_df,
                    target_id=target_id,
                    variables=variables,
                    weights=self.hybrid_weights[target_id],
                    config=self.config,
                )
            )

        elapsed = (time.perf_counter() - started) / 60
        print(
            f"Pipeline fit completed in {elapsed:.2f} minutes."
        )
        return self

    def row_mode(self, row: pd.Series) -> str:
        return (
            "in_domain"
            if str(row.get("country")) in self.seen_countries
            else "out_of_domain"
        )

    def available_prompt_features(
        self,
        row: pd.Series,
        target_id: str,
    ) -> list[str]:
        ranking_order = self.hybrid_rankings[
            target_id
        ]["feature"].tolist()

        ordered = list(
            dict.fromkeys(
                self.hybrid_selected[target_id]
                + ranking_order
            )
        )

        available = [
            variable
            for variable in ordered
            if not pd.isna(row.get(variable))
            and answer_text(variable, row.get(variable)) is not None
        ]

        return available[
            : self.config.prompt_features
        ]

    def _catboost_probs(
        self,
        row: pd.Series,
        target_id: str,
    ) -> dict[str, float]:
        probabilities = (
            self.expert.predict_proba(row, target_id)
            if self.expert.available
            else None
        )

        if probabilities is None:
            probabilities = dict(
                self.priors[target_id]
            )

        total = sum(probabilities.values())
        if total <= 0:
            probabilities = dict(
                self.priors[target_id]
            )
            total = sum(probabilities.values())

        return {
            label: float(
                probabilities.get(label, 0.0)
            ) / total
            for label in labels_for[target_id]
        }

    def _render_examples(
        self,
        examples: list[dict[str, Any]],
        target_id: str,
    ) -> str:
        if not examples:
            return "No retrieved examples used."

        variables = self.hybrid_selected[target_id]
        blocks = []

        for number, example in enumerate(
            examples,
            start=1,
        ):
            evidence = render_evidence(
                example["respondent"],
                variables,
                max_items=self.config.example_features,
            )
            blocks.append(
                f"Example {number} "
                f"(similarity={example['similarity']:.3f}):\n"
                f"{evidence}\n"
                f"Observed target answer: "
                f"{example['target_label']}"
            )

        return "\n\n".join(blocks)

    def build_prediction_prompt(
        self,
        row: pd.Series,
        target_id: str,
        examples: list[dict[str, Any]],
        cb_probs: dict[str, float],
    ) -> str:
        labels = labels_for[target_id]
        features_for_prompt = (
            self.available_prompt_features(
                row,
                target_id,
            )
        )

        labels_block = "\n".join(
            f"- {label}"
            for label in labels
        )
        probability_template = ",\n    ".join(
            f'"{label}": 0.0'
            for label in labels
        )
        prior_text = json.dumps(
            {
                label: round(
                    cb_probs[label],
                    4,
                )
                for label in labels
            },
            ensure_ascii=False,
        )

        label_note = LABEL_NOTES.get(
            target_id,
            "Labels are listed in their official scale order.",
        )

        return f"""
Target-specific reasoning guide:
{TARGET_GUIDES[target_id]}

Respondent country (weak context only):
{row.get('country', 'Not supplied')}

Most important available respondent answers:
{render_evidence(
    row,
    features_for_prompt,
    max_items=self.config.prompt_features,
)}

Two compact labelled analogues from fit data:
{self._render_examples(examples, target_id)}

Auxiliary CatBoost probability prior:
{prior_text}

Target question:
{question_for[target_id]}

Allowed answer labels:
{labels_block}

Label interpretation:
{label_note}

Return JSON only:
{{
  "prediction": "one exact allowed label",
  "probabilities": {{
    {probability_template}
  }}
}}

Requirements:
- Include every allowed label exactly once.
- Probabilities must be numeric, non-negative and sum approximately to 1.
- The prediction must be the label with the largest probability.
- Do not include explanation, markdown or survey codes.
""".strip()

    def build_repair_prompt(
        self,
        target_id: str,
        raw_output: str,
    ) -> str:
        labels = labels_for[target_id]
        labels_block = "\n".join(
            f"- {label}"
            for label in labels
        )
        probability_template = ",\n    ".join(
            f'"{label}": 0.0'
            for label in labels
        )

        return f"""
Reformat the model output below into valid JSON.
Do not reconsider the survey case; preserve its intended classification
when it can be inferred.

Allowed labels:
{labels_block}

Original output:
{raw_output[:2500]}

Return exactly:
{{
  "prediction": "one exact allowed label",
  "probabilities": {{
    {probability_template}
  }}
}}
""".strip()

    def predict_one(
        self,
        row: pd.Series,
        target_id: str,
    ) -> dict[str, Any]:
        row_mode = self.row_mode(row)
        examples = self.retrievers[
            target_id
        ].retrieve(
            row,
            row_mode=row_mode,
        )
        cb_probs = self._catboost_probs(
            row,
            target_id,
        )

        prompt = self.build_prediction_prompt(
            row=row,
            target_id=target_id,
            examples=examples,
            cb_probs=cb_probs,
        )

        raw_first = ""
        raw_repair = ""
        parsed = None
        status = "success_first_pass"
        repair_used = False

        try:
            raw_first = self.llm.complete(
                PREDICTION_SYSTEM,
                prompt,
                thinking=False,
                max_tokens=self.config.first_pass_max_tokens,
            )
            parsed = parse_probability_response(
                raw_first,
                labels_for[target_id],
            )
        except Exception as error:
            raw_first = repr(error)

        if parsed is None:
            repair_used = True
            status = "repair_attempted"
            try:
                raw_repair = self.llm.complete(
                    REPAIR_SYSTEM,
                    self.build_repair_prompt(
                        target_id,
                        raw_first,
                    ),
                    thinking=False,
                    max_tokens=self.config.repair_max_tokens,
                )
                parsed = parse_probability_response(
                    raw_repair,
                    labels_for[target_id],
                )
                if parsed is not None:
                    status = "success_repair"
            except Exception as error:
                raw_repair = repr(error)

        if parsed is None:
            parsed = {
                "prediction": max(
                    cb_probs,
                    key=cb_probs.get,
                ),
                "probabilities": cb_probs,
            }
            status = "catboost_fallback"

        llm_probs = dict(
            parsed["probabilities"]
        )
        blend_weight = (
            self.config.catboost_blend_weight
        )

        final_probs = {
            label: (
                (1.0 - blend_weight)
                * llm_probs.get(label, 0.0)
                + blend_weight
                * cb_probs.get(label, 0.0)
            )
            for label in labels_for[target_id]
        }

        total = sum(final_probs.values())
        final_probs = {
            label: value / total
            for label, value in final_probs.items()
        }

        prediction = max(
            final_probs,
            key=final_probs.get,
        )

        return {
            "respondent_id": row.get("respondent_id"),
            "country": row.get("country"),
            "question_id": target_id,
            "prediction": prediction,
            "probabilities": final_probs,
            "llm_prediction": parsed["prediction"],
            "catboost_prediction": max(
                cb_probs,
                key=cb_probs.get,
            ),
            "row_mode": row_mode,
            "status": status,
            "repair_used": repair_used,
            "second_pass_used": False,
            "margin": probability_margin(
                final_probs
            ),
            "retrieved_ids": [
                example["respondent_id"]
                for example in examples
            ],
            "prompt_features": (
                self.available_prompt_features(
                    row,
                    target_id,
                )
            ),
            "raw_first": raw_first,
            "raw_repair": raw_repair,
        }

    def preview_prompt(
        self,
        row: pd.Series,
        target_id: str,
    ) -> str:
        examples = self.retrievers[
            target_id
        ].retrieve(
            row,
            row_mode=self.row_mode(row),
        )
        cb_probs = self._catboost_probs(
            row,
            target_id,
        )
        return self.build_prediction_prompt(
            row,
            target_id,
            examples,
            cb_probs,
        )


## Block 17. Run predictions in parallel with checkpoints

### Purpose

This cell defines the execution engine that applies the fitted pipeline to many respondent–target pairs efficiently and safely.

### What happens

#### Job creation

For every input respondent, the runner creates one job for each target whose true answer is available for validation or whose answer must be predicted for the official test.

A full official run contains 9,450 jobs.

#### Existing checkpoint loading

Before starting, the function looks for an existing checkpoint associated with `run_id`. Completed respondent–target pairs are loaded and skipped.

This supports resuming after:

- Colab disconnects;
- browser closure;
- temporary API interruption.

#### Parallel execution

Up to `max_workers=48` jobs are sent concurrently using `ThreadPoolExecutor`.

Each completed result contains:

- final prediction;
- LLM and CatBoost predictions;
- probability distributions;
- status;
- whether repair was used;
- probability margin;
- prompt/response diagnostics.

#### Periodic persistence

Every 100 new results, the runner writes a checkpoint. At the end it writes a deduplicated final checkpoint.

#### Output tables

The function returns:

1. a compact predictions table required for submission;
2. a detailed raw/result table for diagnostics and validation.

### Main outputs

`run_fast9_predictions()`.

### API / cost impact

The function definition itself costs nothing. Calling it in Blocks 22, 28 or 29 may create paid LLM requests unless results already exist in the cache/checkpoint.

### What to check

- New job count versus checkpointed job count;
- repair share;
- fallback share;
- total wall time;
- full respondent–target coverage.

Use a new `run_id` and cache path whenever the model or prompt changes.

In [18]:
def run_fast9_predictions(
    pipeline: FastNineCallLLMPipeline,
    input_df: pd.DataFrame,
    run_id: str,
    include_truth: bool,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    truth_df = input_df.copy().reset_index(drop=True)
    masked_df = mask_target_columns(
        truth_df
    ).reset_index(drop=True)

    checkpoint = Path(
        f"fast9_{run_id}_checkpoint.jsonl"
    )
    completed = {}

    if checkpoint.exists():
        with checkpoint.open(
            "r",
            encoding="utf-8",
        ) as file:
            for line in file:
                try:
                    record = json.loads(line)
                    key = (
                        str(record["respondent_id"]),
                        record["question_id"],
                    )
                    completed[key] = record
                except Exception:
                    pass

    truth_lookup = {}
    jobs = []

    for index, row in masked_df.iterrows():
        truth_row = truth_df.iloc[index]
        respondent_id = str(
            row["respondent_id"]
        )

        for target_id in target_ids:
            key = (
                respondent_id,
                target_id,
            )

            if include_truth:
                truth = decode_target_value(
                    target_id,
                    truth_row[target_id],
                )
                if truth is None:
                    continue
                truth_lookup[key] = truth

            if key not in completed:
                jobs.append((
                    row,
                    target_id,
                ))

    print(
        f"New LLM jobs: {len(jobs):,}; "
        f"already checkpointed: {len(completed):,}"
    )

    buffer = []
    buffer_lock = threading.Lock()
    started = time.perf_counter()

    def task(
        row: pd.Series,
        target_id: str,
    ):
        record = pipeline.predict_one(
            row,
            target_id,
        )
        key = (
            str(row["respondent_id"]),
            target_id,
        )
        if include_truth:
            record["truth"] = (
                truth_lookup[key]
            )
        record["probabilities"] = {
            label: float(value)
            for label, value in record[
                "probabilities"
            ].items()
        }
        return record

    def flush_buffer():
        nonlocal buffer
        if not buffer:
            return
        with checkpoint.open(
            "a",
            encoding="utf-8",
        ) as file:
            for item in buffer:
                file.write(
                    json.dumps(
                        item,
                        ensure_ascii=False,
                    )
                    + "\n"
                )
        buffer = []

    with ThreadPoolExecutor(
        max_workers=pipeline.config.max_workers
    ) as pool:
        future_map = {
            pool.submit(
                task,
                row,
                target_id,
            ): (
                str(row["respondent_id"]),
                target_id,
            )
            for row, target_id in jobs
        }

        for future in tqdm(
            as_completed(future_map),
            total=len(future_map),
            desc=run_id,
        ):
            key = future_map[future]

            try:
                record = future.result()
            except Exception as error:
                respondent_id, target_id = key
                row = masked_df[
                    masked_df["respondent_id"]
                    .astype(str)
                    .eq(respondent_id)
                ].iloc[0]

                cb_probs = pipeline._catboost_probs(
                    row,
                    target_id,
                )
                record = {
                    "respondent_id": row[
                        "respondent_id"
                    ],
                    "country": row["country"],
                    "question_id": target_id,
                    "prediction": max(
                        cb_probs,
                        key=cb_probs.get,
                    ),
                    "probabilities": cb_probs,
                    "llm_prediction": None,
                    "catboost_prediction": max(
                        cb_probs,
                        key=cb_probs.get,
                    ),
                    "row_mode": pipeline.row_mode(
                        row
                    ),
                    "status": (
                        "unhandled_error_"
                        "catboost_fallback"
                    ),
                    "repair_used": False,
                    "second_pass_used": False,
                    "margin": probability_margin(
                        cb_probs
                    ),
                    "retrieved_ids": [],
                    "prompt_features": [],
                    "raw_first": repr(error),
                    "raw_repair": "",
                }
                if include_truth:
                    record["truth"] = (
                        truth_lookup[key]
                    )

            completed[key] = record
            buffer.append(record)

            if (
                len(buffer)
                >= pipeline.config.checkpoint_every
            ):
                with buffer_lock:
                    flush_buffer()

    with buffer_lock:
        flush_buffer()

    # Rewrite a clean, deduplicated checkpoint.
    with checkpoint.open(
        "w",
        encoding="utf-8",
    ) as file:
        for key in sorted(completed):
            file.write(
                json.dumps(
                    completed[key],
                    ensure_ascii=False,
                )
                + "\n"
            )

    raw = pd.DataFrame(
        completed.values()
    ).drop_duplicates(
        ["respondent_id", "question_id"],
        keep="last",
    )

    wanted_ids = set(
        truth_df["respondent_id"].astype(str)
    )
    raw = raw[
        raw["respondent_id"]
        .astype(str)
        .isin(wanted_ids)
    ].copy()

    predictions = raw[
        [
            "respondent_id",
            "question_id",
            "prediction",
        ]
    ].copy()

    expected = (
        len(truth_lookup)
        if include_truth
        else len(truth_df) * len(target_ids)
    )

    assert len(raw) == expected, (
        len(raw),
        expected,
    )
    assert not raw.duplicated(
        ["respondent_id", "question_id"]
    ).any()

    for target_id, group in raw.groupby(
        "question_id"
    ):
        assert set(
            group["prediction"]
        ).issubset(
            set(labels_for[target_id])
        )

    elapsed_minutes = (
        time.perf_counter() - started
    ) / 60

    print(
        f"Completed {len(raw):,} predictions "
        f"in {elapsed_minutes:.2f} minutes."
    )
    print(
        "Repair share:",
        f"{raw['repair_used'].mean():.2%}",
    )
    print(
        "Fallback share:",
        f"{raw['status'].str.contains('fallback').mean():.2%}",
    )

    return (
        predictions.reset_index(drop=True),
        raw.reset_index(drop=True),
    )


## Block 18. Define scoring and paired-comparison diagnostics

### Purpose

This cell provides the local evaluation framework used to decide whether Fast9 is strong enough for official inference.

### What happens

The cell defines utilities for:

- balanced respondent sampling;
- local validation execution;
- target-level scoring;
- majority baselines;
- fixed comparison samples;
- paired merging of predictions;
- three-way comparisons;
- gain-retention calculations.

### Main metrics

#### Accuracy

Share of exact respondent-level predictions that are correct.

#### Macro-F1

F1 is calculated independently for every answer label and then averaged equally within each target. The notebook then averages target-level values.

This protects rare answer categories from being ignored.

#### Skill proxy

Accuracy is compared with the target's majority-label accuracy:

\[
\frac{\text{accuracy}-\text{majority share}}
{1-\text{majority share}}.
\]

This mirrors the key leaderboard concept.

#### Alignment proxy

The version in this notebook compares predicted and true aggregate label distributions. It is used for model comparison; the official leaderboard uses an order-aware earth-mover formulation.

#### Coverage

Share of required respondent–target pairs with a prediction.

### Paired diagnostics

The comparison functions evaluate the same respondent–target pairs for two models.

`paired_win_loss_table()` counts:

- new model correct, old model wrong;
- old model correct, new model wrong;
- both correct;
- both wrong.

`paired_cluster_bootstrap()` resamples whole countries rather than individual rows, preserving within-country dependence.

### Main outputs

A collection of scoring and comparison functions used in Blocks 23 and 24.

### API / cost impact

No API call. These functions operate on saved predictions.

### What to check

The team should distinguish:

- respondent-level predictive metrics;
- aggregate-distribution alignment;
- paired evidence;
- uncertainty from country-cluster bootstrap.

Changing metric definitions does not require rerunning LLM inference if row-level predictions have already been saved.

In [19]:
def balanced_respondent_sample(df: pd.DataFrame, n: Optional[int], seed: int) -> pd.DataFrame:
    if n is None or n >= len(df):
        return df.copy().reset_index(drop=True)
    rng = np.random.default_rng(seed)
    countries = list(df["country"].dropna().unique())
    selected = []
    country_indices = {
        c: rng.permutation(df.index[df["country"] == c].to_numpy()).tolist()
        for c in countries
    }
    while len(selected) < n:
        progress = False
        for c in countries:
            if country_indices[c] and len(selected) < n:
                selected.append(country_indices[c].pop())
                progress = True
        if not progress:
            break
    return df.loc[selected].copy().reset_index(drop=True)


def build_profiles_batch(
    pipeline: AlternativeLLMPipeline,
    masked_df: pd.DataFrame,
    workers: Optional[int] = None,
) -> dict[str, str]:
    if not pipeline.config.use_profile_stage:
        return {str(row["respondent_id"]): "Profile stage disabled." for _, row in masked_df.iterrows()}

    workers = workers or pipeline.config.max_workers
    profiles = {}
    with ThreadPoolExecutor(max_workers=workers) as pool:
        future_map = {
            pool.submit(pipeline.get_profile, row): str(row["respondent_id"])
            for _, row in masked_df.iterrows()
        }
        for future in tqdm(as_completed(future_map), total=len(future_map), desc="Profiles"):
            rid = future_map[future]
            try:
                profiles[rid] = future.result()
            except Exception as exc:
                profiles[rid] = f"Profile unavailable: {exc!r}"
    return profiles


def run_local_validation(
    pipeline: AlternativeLLMPipeline,
    valid_df: pd.DataFrame,
    experiment_name: str,
    max_eval_respondents: Optional[int] = 40,
) -> pd.DataFrame:
    eval_truth = balanced_respondent_sample(valid_df, max_eval_respondents, pipeline.config.seed)
    eval_masked = mask_target_columns(eval_truth)
    profiles = build_profiles_batch(pipeline, eval_masked)

    checkpoint = Path(f"validation_{experiment_name}_checkpoint.jsonl")
    completed = {}
    if checkpoint.exists():
        with checkpoint.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    completed[(str(rec["respondent_id"]), rec["question_id"])] = rec
                except Exception:
                    pass

    jobs = []
    truth_lookup = {}
    for idx, row in eval_masked.iterrows():
        truth_row = eval_truth.iloc[idx]
        rid = str(row["respondent_id"])
        for target_id in target_ids:
            truth = decode_target_value(target_id, truth_row[target_id])
            if truth is None:
                continue
            truth_lookup[(rid, target_id)] = truth
            if (rid, target_id) not in completed:
                jobs.append((row, target_id, profiles[rid]))

    new_records = []
    lock = threading.Lock()

    def task(row, target_id, profile):
        rec = pipeline.predict_one(row, target_id, profile=profile)
        rec["truth"] = truth_lookup[(str(row["respondent_id"]), target_id)]
        rec["probabilities"] = {k: float(v) for k, v in rec["probabilities"].items()}
        return rec

    with ThreadPoolExecutor(max_workers=pipeline.config.max_workers) as pool:
        futures = [pool.submit(task, *job) for job in jobs]
        for future in tqdm(as_completed(futures), total=len(futures), desc=experiment_name):
            rec = future.result()
            new_records.append(rec)
            completed[(str(rec["respondent_id"]), rec["question_id"])] = rec
            if len(new_records) % pipeline.config.checkpoint_every == 0:
                with lock, checkpoint.open("a", encoding="utf-8") as f:
                    for item in new_records[-pipeline.config.checkpoint_every:]:
                        f.write(json.dumps(item, ensure_ascii=False) + "\n")

    with checkpoint.open("w", encoding="utf-8") as f:
        for rec in completed.values():
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    result = pd.DataFrame(completed.values())
    expected_ids = set(eval_truth["respondent_id"].astype(str))
    result = result[result["respondent_id"].astype(str).isin(expected_ids)].copy()
    return result.reset_index(drop=True)


def score_predictions(predictions: pd.DataFrame, majority: dict[str, str], prediction_col: str = "prediction"):
    rows = []
    for target_id in target_ids:
        g = predictions[predictions["question_id"] == target_id].copy()
        if g.empty:
            continue
        valid_mask = g[prediction_col].isin(labels_for[target_id])
        coverage = float(valid_mask.mean())
        y_true = g["truth"].astype(str)
        y_pred = g[prediction_col].fillna("__INVALID__").astype(str)

        accuracy = float(accuracy_score(y_true, y_pred))
        macro_f1 = float(f1_score(
            y_true,
            y_pred,
            labels=labels_for[target_id],
            average="macro",
            zero_division=0,
        ))
        majority_accuracy = float((y_true == majority[target_id]).mean())
        skill_proxy = (
            (accuracy - majority_accuracy) / (1 - majority_accuracy)
            if majority_accuracy < 1 else 0.0
        )

        true_dist = y_true.value_counts(normalize=True).reindex(labels_for[target_id], fill_value=0.0)
        pred_dist = y_pred.value_counts(normalize=True).reindex(labels_for[target_id], fill_value=0.0)
        alignment_proxy = float(1 - 0.5 * np.abs(true_dist - pred_dist).sum())

        rows.append({
            "question_id": target_id,
            "n": len(g),
            "coverage": coverage,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "majority_accuracy": majority_accuracy,
            "skill_proxy": skill_proxy,
            "alignment_proxy": alignment_proxy,
        })

    per_target = pd.DataFrame(rows)
    overall = pd.DataFrame([{
        "question_id": "MEAN_OVER_TARGETS",
        "n": int(per_target["n"].sum()),
        "coverage": per_target["coverage"].mean(),
        "accuracy": per_target["accuracy"].mean(),
        "macro_f1": per_target["macro_f1"].mean(),
        "majority_accuracy": per_target["majority_accuracy"].mean(),
        "skill_proxy": per_target["skill_proxy"].mean(),
        "alignment_proxy": per_target["alignment_proxy"].mean(),
    }])
    return pd.concat([per_target, overall], ignore_index=True)


def majority_baseline_frame(valid_df: pd.DataFrame, majority: dict[str, str], max_n: Optional[int], seed: int):
    sampled = balanced_respondent_sample(valid_df, max_n, seed)
    records = []
    for _, row in sampled.iterrows():
        for target_id in target_ids:
            truth = decode_target_value(target_id, row[target_id])
            if truth is None:
                continue
            records.append({
                "respondent_id": row["respondent_id"],
                "question_id": target_id,
                "truth": truth,
                "prediction": majority[target_id],
            })
    return pd.DataFrame(records)


def config_signature(config: PipelineConfig, prefix: str) -> str:
    payload = asdict(config)
    payload.pop("cache_path", None)
    text = json.dumps(
        {"prefix": prefix, "config": payload},
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:10]


def make_fixed_comparison_sample(
    valid_df: pd.DataFrame,
    n_respondents: Optional[int],
    seed: int,
) -> pd.DataFrame:
    """Choose respondents once, then pass exactly this frame to both methods."""
    sample = balanced_respondent_sample(
        valid_df,
        n=n_respondents,
        seed=seed,
    )
    assert sample["respondent_id"].is_unique
    return sample.reset_index(drop=True)


def paired_prediction_frame(
    legacy_result: pd.DataFrame,
    alternative_result: pd.DataFrame,
) -> pd.DataFrame:
    keys = ["respondent_id", "question_id"]

    old = legacy_result[
        keys + ["country", "truth", "prediction", "status"]
    ].rename(columns={
        "prediction": "legacy_prediction",
        "status": "legacy_status",
    })

    new = alternative_result[
        keys + ["truth", "prediction", "status", "second_pass_used"]
    ].rename(columns={
        "truth": "alternative_truth",
        "prediction": "alternative_prediction",
        "status": "alternative_status",
    })

    paired = old.merge(
        new,
        on=keys,
        how="inner",
        validate="one_to_one",
    )

    if len(paired) != len(old) or len(paired) != len(new):
        raise AssertionError(
            "The two methods were not evaluated on exactly the same "
            "respondent-target pairs."
        )

    if not (
        paired["truth"].astype(str)
        == paired["alternative_truth"].astype(str)
    ).all():
        raise AssertionError("Truth labels differ between methods.")

    paired = paired.drop(columns="alternative_truth")
    paired["legacy_correct"] = (
        paired["legacy_prediction"] == paired["truth"]
    )
    paired["alternative_correct"] = (
        paired["alternative_prediction"] == paired["truth"]
    )
    paired["alternative_win"] = (
        paired["alternative_correct"]
        & ~paired["legacy_correct"]
    )
    paired["alternative_loss"] = (
        paired["legacy_correct"]
        & ~paired["alternative_correct"]
    )
    return paired


def compare_score_tables(
    legacy_result: pd.DataFrame,
    alternative_result: pd.DataFrame,
    legacy_majority: dict[str, str],
    alternative_majority: dict[str, str],
) -> pd.DataFrame:
    old_score = score_predictions(
        legacy_result,
        legacy_majority,
    ).rename(columns={
        column: f"legacy_{column}"
        for column in [
            "n",
            "coverage",
            "accuracy",
            "macro_f1",
            "majority_accuracy",
            "skill_proxy",
            "alignment_proxy",
        ]
    })

    new_score = score_predictions(
        alternative_result,
        alternative_majority,
    ).rename(columns={
        column: f"alternative_{column}"
        for column in [
            "n",
            "coverage",
            "accuracy",
            "macro_f1",
            "majority_accuracy",
            "skill_proxy",
            "alignment_proxy",
        ]
    })

    comparison = old_score.merge(
        new_score,
        on="question_id",
        how="inner",
        validate="one_to_one",
    )

    for metric in [
        "coverage",
        "accuracy",
        "macro_f1",
        "skill_proxy",
        "alignment_proxy",
    ]:
        comparison[f"delta_{metric}"] = (
            comparison[f"alternative_{metric}"]
            - comparison[f"legacy_{metric}"]
        )

    return comparison


def _mean_target_macro_f1(
    frame: pd.DataFrame,
    prediction_col: str,
) -> float:
    scores = []
    for target_id in target_ids:
        group = frame[frame["question_id"] == target_id]
        if group.empty:
            continue
        scores.append(
            f1_score(
                group["truth"].astype(str),
                group[prediction_col].astype(str),
                labels=labels_for[target_id],
                average="macro",
                zero_division=0,
            )
        )
    return float(np.mean(scores)) if scores else float("nan")


def _mean_target_alignment(
    frame: pd.DataFrame,
    prediction_col: str,
) -> float:
    values = []
    for target_id in target_ids:
        group = frame[frame["question_id"] == target_id]
        if group.empty:
            continue
        labels = labels_for[target_id]
        truth_dist = (
            group["truth"]
            .astype(str)
            .value_counts(normalize=True)
            .reindex(labels, fill_value=0.0)
        )
        pred_dist = (
            group[prediction_col]
            .astype(str)
            .value_counts(normalize=True)
            .reindex(labels, fill_value=0.0)
        )
        values.append(
            1.0 - 0.5 * np.abs(truth_dist - pred_dist).sum()
        )
    return float(np.mean(values)) if values else float("nan")


def paired_cluster_bootstrap(
    paired: pd.DataFrame,
    cluster_col: str = "country",
    n_bootstrap: int = 1000,
    seed: int = 42,
) -> pd.DataFrame:
    """
    Bootstrap deltas while preserving all rows inside a country cluster.

    The reported alignment is the notebook's local distribution-alignment
    proxy, not necessarily the hidden leaderboard implementation.
    """
    required = {
        cluster_col,
        "question_id",
        "truth",
        "legacy_prediction",
        "alternative_prediction",
    }
    missing = required - set(paired.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")

    clusters = paired[cluster_col].dropna().astype(str).unique()
    if len(clusters) < 2:
        raise ValueError("At least two bootstrap clusters are required.")

    rng = np.random.default_rng(seed)
    boot_rows = []

    for _ in range(n_bootstrap):
        sampled_clusters = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )
        sampled = pd.concat(
            [
                paired[
                    paired[cluster_col].astype(str) == cluster
                ]
                for cluster in sampled_clusters
            ],
            ignore_index=True,
        )

        legacy_f1 = _mean_target_macro_f1(
            sampled,
            "legacy_prediction",
        )
        alternative_f1 = _mean_target_macro_f1(
            sampled,
            "alternative_prediction",
        )

        legacy_alignment = _mean_target_alignment(
            sampled,
            "legacy_prediction",
        )
        alternative_alignment = _mean_target_alignment(
            sampled,
            "alternative_prediction",
        )

        legacy_accuracy = float(
            (
                sampled["legacy_prediction"]
                == sampled["truth"]
            ).mean()
        )
        alternative_accuracy = float(
            (
                sampled["alternative_prediction"]
                == sampled["truth"]
            ).mean()
        )

        boot_rows.append({
            "delta_macro_f1": alternative_f1 - legacy_f1,
            "delta_alignment_proxy": (
                alternative_alignment - legacy_alignment
            ),
            "delta_accuracy": (
                alternative_accuracy - legacy_accuracy
            ),
        })

    boot = pd.DataFrame(boot_rows)

    observed = {
        "delta_macro_f1": (
            _mean_target_macro_f1(
                paired,
                "alternative_prediction",
            )
            - _mean_target_macro_f1(
                paired,
                "legacy_prediction",
            )
        ),
        "delta_alignment_proxy": (
            _mean_target_alignment(
                paired,
                "alternative_prediction",
            )
            - _mean_target_alignment(
                paired,
                "legacy_prediction",
            )
        ),
        "delta_accuracy": float(
            (
                paired["alternative_prediction"]
                == paired["truth"]
            ).mean()
            - (
                paired["legacy_prediction"]
                == paired["truth"]
            ).mean()
        ),
    }

    summary = []
    for metric, estimate in observed.items():
        values = boot[metric].to_numpy()
        summary.append({
            "metric": metric,
            "observed_delta": estimate,
            "ci_2_5": float(np.quantile(values, 0.025)),
            "ci_97_5": float(np.quantile(values, 0.975)),
            "bootstrap_probability_delta_gt_0": float(
                np.mean(values > 0)
            ),
        })

    return pd.DataFrame(summary)


def paired_win_loss_table(
    paired: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for target_id in target_ids + ["ALL_TARGETS"]:
        group = (
            paired
            if target_id == "ALL_TARGETS"
            else paired[paired["question_id"] == target_id]
        )
        if group.empty:
            continue

        wins = int(group["alternative_win"].sum())
        losses = int(group["alternative_loss"].sum())
        ties_both_correct = int(
            (
                group["legacy_correct"]
                & group["alternative_correct"]
            ).sum()
        )
        ties_both_wrong = int(
            (
                ~group["legacy_correct"]
                & ~group["alternative_correct"]
            ).sum()
        )

        rows.append({
            "question_id": target_id,
            "n": len(group),
            "alternative_wins": wins,
            "alternative_losses": losses,
            "net_wins": wins - losses,
            "both_correct": ties_both_correct,
            "both_wrong": ties_both_wrong,
        })

    return pd.DataFrame(rows)


def three_way_score_table(
    legacy_result: pd.DataFrame,
    full_alternative_result: pd.DataFrame,
    fast9_result: pd.DataFrame,
    majority: dict[str, str],
) -> pd.DataFrame:
    frames = []

    for method_name, result in [
        ("legacy", legacy_result),
        ("full_alternative", full_alternative_result),
        ("fast9", fast9_result),
    ]:
        score = score_predictions(
            result,
            majority,
        )
        score = score.rename(columns={
            column: f"{method_name}_{column}"
            for column in [
                "n",
                "coverage",
                "accuracy",
                "macro_f1",
                "majority_accuracy",
                "skill_proxy",
                "alignment_proxy",
            ]
        })
        frames.append(score)

    combined = frames[0]
    for frame in frames[1:]:
        combined = combined.merge(
            frame,
            on="question_id",
            how="inner",
            validate="one_to_one",
        )

    for metric in [
        "coverage",
        "accuracy",
        "macro_f1",
        "skill_proxy",
        "alignment_proxy",
    ]:
        combined[
            f"fast9_minus_legacy_{metric}"
        ] = (
            combined[f"fast9_{metric}"]
            - combined[f"legacy_{metric}"]
        )
        combined[
            f"fast9_minus_full_{metric}"
        ] = (
            combined[f"fast9_{metric}"]
            - combined[
                f"full_alternative_{metric}"
            ]
        )

    return combined


def gain_retention(
    legacy_value: float,
    full_value: float,
    fast_value: float,
) -> float:
    denominator = full_value - legacy_value
    if denominator <= 0:
        return float("nan")
    return (
        fast_value - legacy_value
    ) / denominator


# Part II. Validate Fast9 against the saved OOD pilot

The first pipeline and the full alternative pipeline were already run on a fixed set of 75 OOD respondents. Their predictions are stored in a ZIP archive.

This section:

1. loads those saved predictions;
2. reconstructs the exact same train/validation split;
3. fits Fast9 on the same fit fold;
4. predicts the same respondents;
5. compares all three approaches on identical respondent–target pairs.

This design avoids spending LLM budget rerunning the two reference systems.

## Block 19. Upload and read the saved OOD pilot archive

### Purpose

This cell loads the fixed comparison sample and the predictions produced by the earlier two pipelines.

### What happens

1. Colab opens a file-upload dialog.
2. The first uploaded ZIP file is extracted into `/content/fast9_reference`.
3. The code searches recursively for:
   - `sample.csv`;
   - `legacy_result.csv`;
   - `alternative_result.csv`.
4. Respondent IDs are converted to strings for stable joins.
5. The cell prints sample sizes.
6. It verifies that legacy and full-alternative result tables cover exactly the same respondent–target pairs.

### Main outputs

- `reference_sample`;
- `reference_legacy`;
- `reference_full`.

### API / cost impact

No LLM call.

### What to check

Upload the correct `pilot_ood_results_*.zip`.

Expected broad values:

- 75 unique reference respondents;
- approximately 650 valid respondent–target pairs, because some true targets are missing;
- identical pair sets for the two saved models.

A mismatch means the archive cannot support a valid paired comparison.

In [20]:
from google.colab import files

uploaded = files.upload()

zip_names = [
    name
    for name in uploaded
    if name.lower().endswith(".zip")
]

if not zip_names:
    raise ValueError(
        "Upload pilot_ood_results_*.zip"
    )

REFERENCE_DIR = Path(
    "/content/fast9_reference"
)
shutil.rmtree(
    REFERENCE_DIR,
    ignore_errors=True,
)
REFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    zip_names[0],
    "r",
) as archive:
    archive.extractall(
        REFERENCE_DIR
    )

def find_required_file(
    filename: str,
) -> Path:
    matches = list(
        REFERENCE_DIR.rglob(filename)
    )
    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found "
            f"in {zip_names[0]}"
        )
    return matches[0]


reference_sample = pd.read_csv(
    find_required_file("sample.csv")
)
reference_legacy = pd.read_csv(
    find_required_file("legacy_result.csv")
)
reference_full = pd.read_csv(
    find_required_file("alternative_result.csv")
)

for frame in [
    reference_sample,
    reference_legacy,
    reference_full,
]:
    frame["respondent_id"] = (
        frame["respondent_id"].astype(str)
    )

print(
    "Reference respondents:",
    reference_sample[
        "respondent_id"
    ].nunique(),
)
print(
    "Legacy pairs:",
    len(reference_legacy),
)
print(
    "Full alternative pairs:",
    len(reference_full),
)

assert set(
    reference_legacy[
        ["respondent_id", "question_id"]
    ].itertuples(index=False, name=None)
) == set(
    reference_full[
        ["respondent_id", "question_id"]
    ].itertuples(index=False, name=None)
)


Saving pilot_ood_results_20260714_213132.zip to pilot_ood_results_20260714_213132.zip
Reference respondents: 75
Legacy pairs: 650
Full alternative pairs: 650


## Block 20. Reconstruct the OOD fold and fit Fast9

### Purpose

This cell ensures that Fast9 is trained on exactly the same fit data used to create the saved OOD pilot comparison.

### What happens

1. It reconstructs the seed-42, 15-country OOD split.
2. It checks that every respondent in `reference_sample` belongs to the reconstructed validation fold.
3. It creates a separate validation configuration and cache.
4. It wraps the Nebius client in the cache-aware client.
5. It fits the full Fast9 statistical pipeline on `ood_split.fit`:
   - NMI/mRMR selector;
   - fit-only priors;
   - nine CatBoost models;
   - hybrid feature rankings;
   - nine retrieval indices.
6. It displays the selected target-specific feature plan.

### Main outputs

- `validation_cfg`;
- `validation_llm`;
- `fast9_validation_pipeline`.

### API / cost impact

No LLM inference occurs. This cell may take a few minutes for feature selection and CatBoost fitting but consumes no Nebius tokens.

### What to check

The split-match assertion must pass.

In the displayed feature table, review:

- selected features per target;
- protected mRMR features;
- CatBoost additions;
- selector and CatBoost importance;
- final hybrid scores.

This is the primary audit point for understanding what evidence each target will use.

In [21]:
ood_split = make_out_domain_split(
    train,
    n_countries=CFG.out_domain_n_countries,
    val_per_country=CFG.out_domain_val_per_country,
    seed=CFG.seed,
)

reference_ids = set(
    reference_sample[
        "respondent_id"
    ].astype(str)
)
valid_ids = set(
    ood_split.valid[
        "respondent_id"
    ].astype(str)
)

missing_reference_ids = (
    reference_ids - valid_ids
)
if missing_reference_ids:
    raise AssertionError(
        "The reconstructed split does not "
        "match the saved pilot sample. "
        f"Missing IDs: "
        f"{list(missing_reference_ids)[:5]}"
    )

validation_cfg = copy.deepcopy(
    CFG
)
validation_cfg.cache_path = (
    "llm_cache_fast9_validation.jsonl"
)

validation_llm = CachedNebiusClient(
    validation_cfg,
    api_key=nebius_key,
    client=raw_nebius_client,
)

fast9_validation_pipeline = (
    FastNineCallLLMPipeline(
        validation_cfg,
        validation_llm,
    )
    .fit(
        ood_split.fit,
        mode=ood_split.mode,
    )
)

display(
    fast9_validation_pipeline
    .feature_importance_table
    .query(
        "selected_for_prompt_or_retrieval"
    )
    .groupby("question_id")
    .head(16)[
        [
            "question_id",
            "feature",
            "question",
            "protected_mrmr",
            "catboost_top_addition",
            "adjusted_score",
            "catboost_importance",
            "hybrid_score",
        ]
    ]
)


Loaded 0 cached LLM responses


Encoding feature columns:   0%|          | 0/278 [00:00<?, ?it/s]

Target feature selection:   0%|          | 0/9 [00:00<?, ?it/s]

Pipeline fit completed in 2.05 minutes.


,question_id,feature,question,protected_mrmr,catboost_top_addition,adjusted_score,catboost_importance,hybrid_score
0,Q201,Q203,How often do you use radio news to obtain information about this country and the world?,True,True,0.073650,1.000000,0.935882
1,Q201,Q223,"If there were a national election tomorrow, for which party would you vote? If you do not know, which party appeals ...",True,True,0.081028,0.539691,0.882578
2,Q201,N_REGION_ISO,Which region do you currently live in?,False,False,0.079545,0.000000,0.784106
3,Q201,Q202,How often do you use tv news to obtain information about this country and the world?,True,True,0.057734,0.617523,0.722152
4,Q201,Q290,How do you identify when it comes to Racial belonging/ ethnic group in your country?,False,False,0.067510,0.000000,0.668043
5,Q201,Q268,In which country was your father born? List of codes in Annex,False,False,0.062825,0.000000,0.632063
6,Q201,Q267,In which country was your mother born?,False,False,0.062681,0.000000,0.630961
7,Q201,Q266,In which country were you born?,False,False,0.061474,0.000000,0.627932
8,Q201,Q272,What language do you normally speak at home?,False,False,0.060843,0.000000,0.622513
9,Q201,Q205,How often do you use email to obtain information about this country and the world?,True,True,0.046991,0.504285,0.605829


## Block 21. Preview one complete prompt

### Purpose

Before spending tokens, this cell shows exactly what the LLM would receive for one respondent and one target.

### What happens

1. It copies the fixed validation sample.
2. It masks all target columns.
3. It selects the first respondent.
4. It builds the Q17 prompt using the fitted pipeline.
5. It prints the prompt without calling the LLM.

The preview normally contains:

- target-specific guide;
- respondent country/context according to this notebook version;
- direct decoded respondent answers;
- two retrieved labelled examples;
- CatBoost prior;
- exact target question;
- exact official labels;
- requested JSON format.

### Main outputs

A human-readable prompt printed to the notebook.

### API / cost impact

No API call.

### What to check

Review this cell carefully as a team:

- Are all direct answers permitted features?
- Are target columns absent?
- Are decoded values understandable?
- Are retrieved examples plausible and not duplicated?
- Are official labels complete and exact?
- Is the prompt concise enough?
- Does the guide help without dictating the answer?

Do not tune the model from a single prompt unless a genuine structural problem is visible.

In [22]:
preview_row = mask_target_columns(
    reference_sample
).iloc[0]

print(
    fast9_validation_pipeline.preview_prompt(
        preview_row,
        "Q17",
    )
)


Target-specific reasoning guide:
Infer preference for child obedience versus autonomy. Consider religion, authority orientation, family values, traditionalism and whether independence is valued for children.

Respondent country (weak context only):
Bolivia

Most important available respondent answers:
- [Q290] How do you identify when it comes to Racial belonging/ ethnic group in your country?
  Answer: BO: Not pertaining to Indigenous groups
- [Q8] Do you consider independence to be an especially important quality for children to learn at home?
  Answer: Not so important
- [Q45] If society placed more emphasis on developing technology in the future, would you consider that a good thing, a bad thing, or wouldn't you mind?
  Answer: Good thing
- [Q57] Generally speaking, would you say that most people can be trusted or that you need to be very careful in dealing with people?
  Answer: Need to be very careful
- [Q11] Do you consider imagination to be an especially important quality for c

## Block 22. Run Fast9 on the fixed OOD validation sample

### Purpose

This is the first cell that performs a substantial batch of paid LLM inference for the new model.

### What happens

`run_fast9_predictions()` is called on the same 75 respondents used by the saved reference pipelines.

Because some respondents lack truth for some target questions, the run usually contains around 650 rather than 675 scored jobs.

Each job produces:

- LLM probability distribution;
- CatBoost probability distribution;
- blended prediction;
- status and repair/fallback information;
- truth label for later scoring.

The cell records total wall time.

### Main outputs

- `fast9_validation_predictions` — compact predictions;
- `fast9_validation_result` — detailed predictions with truth and diagnostics;
- printed validation runtime.

### API / cost impact

Yes. This cell sends new LLM requests unless they already exist in the validation cache/checkpoint.

With 48 workers, the run has typically completed in a few minutes.

### What to check

At the start:

```text
New LLM jobs: ...
already checkpointed: ...
```

For a genuinely new model version, expect zero checkpointed jobs.

At the end, review:

- completion count;
- runtime;
- repair share;
- fallback share.

Zero repair and fallback indicate clean structured outputs, but non-zero values are acceptable if coverage remains complete and the rates are small.

In [23]:
validation_started = time.perf_counter()

fast9_validation_predictions, fast9_validation_result = (
    run_fast9_predictions(
        pipeline=fast9_validation_pipeline,
        input_df=reference_sample,
        run_id="validation_ood_seed42_n75",
        include_truth=True,
    )
)

validation_elapsed = (
    time.perf_counter()
    - validation_started
) / 60

print(
    f"Total validation wall time: "
    f"{validation_elapsed:.2f} minutes."
)


New LLM jobs: 650; already checkpointed: 0


validation_ood_seed42_n75:   0%|          | 0/650 [00:00<?, ?it/s]

Completed 650 predictions in 3.94 minutes.
Repair share: 0.00%
Fallback share: 0.00%
Total validation wall time: 3.94 minutes.


## Block 23. Compare legacy, full alternative and Fast9

### Purpose

This cell answers whether Fast9 is materially better than the original submission and whether it preserves the gains of the slower full alternative pipeline.

### What happens

#### Three-way target table

For every target and for the average across targets, the code reports:

- Macro-F1;
- accuracy;
- alignment proxy;
- coverage;
- Fast9 minus legacy;
- Fast9 minus full alternative.

#### Gain retention

For each main metric:

\[
\text{retained share}
=
\frac{\text{Fast9}-\text{legacy}}
{\text{full alternative}-\text{legacy}}.
\]

A value:

- below 1 means Fast9 retains part of the full gain;
- equal to 1 means it matches the full alternative;
- above 1 means its point estimate exceeds the full alternative.

#### Paired comparison against legacy

The code joins predictions on identical respondent–target pairs, then calculates:

- country-cluster bootstrap confidence intervals;
- probability that the delta is positive;
- wins, losses and net wins.

#### Paired comparison against the full alternative

The same diagnostics are repeated using the full alternative as the reference.

### Main outputs

- `score_comparison`;
- `overall`;
- `retention_summary`;
- paired frames;
- bootstrap tables;
- win/loss tables.

### API / cost impact

No API requests. This cell uses already generated predictions.

### What to check

The most important evidence is:

- mean Macro-F1 gain versus legacy;
- Skill/accuracy direction;
- critical-target performance;
- bootstrap interval relative to zero;
- paired net wins;
- whether any gain depends on a single target.

A small point-estimate advantage over the full alternative should not be described as statistically certain if its interval crosses zero.

In [24]:
score_comparison = three_way_score_table(
    legacy_result=reference_legacy,
    full_alternative_result=reference_full,
    fast9_result=fast9_validation_result,
    majority=fast9_validation_pipeline.majority,
)

display(
    score_comparison[
        [
            "question_id",
            "legacy_macro_f1",
            "full_alternative_macro_f1",
            "fast9_macro_f1",
            "fast9_minus_legacy_macro_f1",
            "fast9_minus_full_macro_f1",
            "legacy_accuracy",
            "full_alternative_accuracy",
            "fast9_accuracy",
            "legacy_alignment_proxy",
            "full_alternative_alignment_proxy",
            "fast9_alignment_proxy",
            "fast9_coverage",
        ]
    ]
)

overall = (
    score_comparison
    .query(
        "question_id == 'MEAN_OVER_TARGETS'"
    )
    .iloc[0]
)

retention_summary = pd.DataFrame([
    {
        "metric": metric,
        "legacy": float(
            overall[f"legacy_{metric}"]
        ),
        "full_alternative": float(
            overall[
                f"full_alternative_{metric}"
            ]
        ),
        "fast9": float(
            overall[f"fast9_{metric}"]
        ),
        "fast9_minus_legacy": float(
            overall[
                f"fast9_minus_legacy_{metric}"
            ]
        ),
        "retained_share_of_full_gain": (
            gain_retention(
                float(
                    overall[
                        f"legacy_{metric}"
                    ]
                ),
                float(
                    overall[
                        f"full_alternative_{metric}"
                    ]
                ),
                float(
                    overall[
                        f"fast9_{metric}"
                    ]
                ),
            )
        ),
    }
    for metric in [
        "macro_f1",
        "accuracy",
        "alignment_proxy",
    ]
])

display(retention_summary)


paired_vs_legacy = paired_prediction_frame(
    reference_legacy,
    fast9_validation_result,
)
bootstrap_vs_legacy = paired_cluster_bootstrap(
    paired_vs_legacy,
    cluster_col="country",
    n_bootstrap=1000,
    seed=CFG.seed,
)
win_loss_vs_legacy = paired_win_loss_table(
    paired_vs_legacy
)

paired_vs_full = paired_prediction_frame(
    reference_full,
    fast9_validation_result,
)
bootstrap_vs_full = paired_cluster_bootstrap(
    paired_vs_full,
    cluster_col="country",
    n_bootstrap=1000,
    seed=CFG.seed,
)
win_loss_vs_full = paired_win_loss_table(
    paired_vs_full
)

print("FAST9 versus first pipeline")
display(bootstrap_vs_legacy)
display(win_loss_vs_legacy)

print("FAST9 versus full alternative")
display(bootstrap_vs_full)
display(win_loss_vs_full)


,question_id,legacy_macro_f1,full_alternative_macro_f1,fast9_macro_f1,fast9_minus_legacy_macro_f1,fast9_minus_full_macro_f1,legacy_accuracy,full_alternative_accuracy,fast9_accuracy,legacy_alignment_proxy,full_alternative_alignment_proxy,fast9_alignment_proxy,fast9_coverage
0,Q201,0.321391,0.332393,0.323571,0.002180,-0.008822,0.391892,0.391892,0.391892,0.729730,0.770270,0.891892,1.0
1,Q73,0.697922,0.768931,0.775659,0.077736,0.006728,0.785714,0.828571,0.800000,0.885714,0.900000,0.900000,1.0
2,Q227,0.498685,0.526710,0.523864,0.025180,-0.002846,0.507246,0.536232,0.536232,0.869565,0.869565,0.855072,1.0
3,Q209,0.681455,0.654762,0.705671,0.024216,0.050909,0.690141,0.676056,0.732394,0.816901,0.802817,0.887324,1.0
4,Q33,0.390222,0.449841,0.410610,0.020388,-0.039231,0.465753,0.465753,0.438356,0.630137,0.753425,0.712329,1.0
5,Q148,0.668700,0.686720,0.734144,0.065444,0.047424,0.693333,0.706667,0.746667,0.866667,0.906667,0.933333,1.0
6,Q17,0.400000,0.638796,0.649464,0.249464,0.010668,0.416667,0.666667,0.652778,0.527778,0.972222,0.791667,1.0
7,Q186,0.508362,0.674881,0.694622,0.186260,0.019741,0.589041,0.698630,0.726027,0.753425,0.876712,0.904110,1.0
8,Q242,0.140316,0.328432,0.335705,0.195389,0.007273,0.328767,0.397260,0.452055,0.328767,0.767123,0.821918,1.0
9,MEAN_OVER_TARGETS,0.478561,0.562385,0.572590,0.094028,0.010205,0.540951,0.596414,0.608489,0.712076,0.846534,0.855294,1.0


,metric,legacy,full_alternative,fast9,fast9_minus_legacy,retained_share_of_full_gain
0,macro_f1,0.478561,0.562385,0.572590,0.094028,1.121741
1,accuracy,0.540951,0.596414,0.608489,0.067538,1.217704
2,alignment_proxy,0.712076,0.846534,0.855294,0.143218,1.065153


FAST9 versus first pipeline


,metric,observed_delta,ci_2_5,ci_97_5,bootstrap_probability_delta_gt_0
0,delta_macro_f1,0.094028,0.050371,0.125475,1.000
1,delta_alignment_proxy,0.143218,0.097507,0.168535,1.000
2,delta_accuracy,0.067692,0.025991,0.104040,0.999


,question_id,n,alternative_wins,alternative_losses,net_wins,both_correct,both_wrong
0,Q201,74,7,7,0,22,38
1,Q73,70,5,4,1,51,10
2,Q227,69,4,2,2,33,30
3,Q209,71,7,4,3,45,15
4,Q33,73,3,5,-2,29,36
5,Q148,75,6,2,4,50,17
6,Q17,72,21,4,17,26,21
7,Q186,73,12,2,10,41,18
8,Q242,73,16,7,9,17,33
9,ALL_TARGETS,650,81,37,44,314,218


FAST9 versus full alternative


,metric,observed_delta,ci_2_5,ci_97_5,bootstrap_probability_delta_gt_0
0,delta_macro_f1,0.010205,-0.003883,0.024044,0.913
1,delta_alignment_proxy,0.008760,-0.009436,0.040641,0.855
2,delta_accuracy,0.012308,0.003058,0.026481,0.999


,question_id,n,alternative_wins,alternative_losses,net_wins,both_correct,both_wrong
0,Q201,74,5,5,0,24,40
1,Q73,70,1,3,-2,55,11
2,Q227,69,0,0,0,37,32
3,Q209,71,7,3,4,45,16
4,Q33,73,3,5,-2,29,36
5,Q148,75,3,0,3,53,19
6,Q17,72,8,9,-1,39,16
7,Q186,73,3,1,2,50,19
8,Q242,73,8,4,4,25,36
9,ALL_TARGETS,650,38,30,8,357,225


## Block 24. Apply an automatic go/no-go rule

### Purpose

This cell converts the validation evidence into a transparent decision about whether to spend the larger official-test budget.

### What happens

Fast9 proceeds only if all seven conditions are satisfied:

1. mean Macro-F1 improves by at least 0.02 over legacy;
2. accuracy is not below legacy;
3. alignment is not below legacy;
4. coverage is 100%;
5. at least 80% of the full alternative's F1 gain is retained;
6. Q17, Q186 and Q242 all improve over legacy;
7. paired net wins against legacy are positive.

The code displays each criterion separately and prints:

- F1 gain retention;
- either `PROCEED TO OFFICIAL TEST`;
- or `DO NOT RUN OFFICIAL TEST YET`.

### Main outputs

- `decision_criteria`;
- `f1_retention`;
- `FAST9_DECISION`.

### API / cost impact

No API call.

### What to check

This rule is deliberately conservative. The team should review both:

- whether every implemented condition passes;
- whether the conditions themselves still reflect the agreed decision policy.

Do not override a failed condition without understanding why it failed.

In [25]:
critical_targets = [
    "Q17",
    "Q186",
    "Q242",
]

critical_target_rows = (
    score_comparison
    .set_index("question_id")
    .loc[critical_targets]
)

f1_retention = gain_retention(
    float(
        overall["legacy_macro_f1"]
    ),
    float(
        overall[
            "full_alternative_macro_f1"
        ]
    ),
    float(
        overall["fast9_macro_f1"]
    ),
)

all_wins = (
    win_loss_vs_legacy
    .query(
        "question_id == 'ALL_TARGETS'"
    )
    .iloc[0]
)

decision_criteria = {
    "macro_f1_delta_vs_legacy_at_least_0_02": (
        float(
            overall[
                "fast9_minus_legacy_macro_f1"
            ]
        )
        >= 0.02
    ),
    "accuracy_not_below_legacy": (
        float(
            overall[
                "fast9_minus_legacy_accuracy"
            ]
        )
        >= 0.0
    ),
    "alignment_not_below_legacy": (
        float(
            overall[
                "fast9_minus_legacy_alignment_proxy"
            ]
        )
        >= 0.0
    ),
    "coverage_is_100_percent": (
        float(
            overall["fast9_coverage"]
        )
        >= 1.0
    ),
    "retains_at_least_80_percent_of_full_f1_gain": (
        not pd.isna(f1_retention)
        and f1_retention >= 0.80
    ),
    "critical_targets_all_improve_over_legacy": (
        critical_target_rows[
            "fast9_minus_legacy_macro_f1"
        ].gt(0).all()
    ),
    "paired_net_wins_vs_legacy_positive": (
        int(all_wins["net_wins"])
        > 0
    ),
}

FAST9_DECISION = (
    "PROCEED TO OFFICIAL TEST"
    if all(decision_criteria.values())
    else "DO NOT RUN OFFICIAL TEST YET"
)

display(pd.DataFrame([
    {
        "criterion": name,
        "passed": bool(passed),
    }
    for name, passed
    in decision_criteria.items()
]))

print(
    "F1 gain retention:",
    (
        f"{f1_retention:.1%}"
        if not pd.isna(f1_retention)
        else "not defined"
    ),
)
print(FAST9_DECISION)


,criterion,passed
0,macro_f1_delta_vs_legacy_at_least_0_02,True
1,accuracy_not_below_legacy,True
2,alignment_not_below_legacy,True
3,coverage_is_100_percent,True
4,retains_at_least_80_percent_of_full_f1_gain,True
5,critical_targets_all_improve_over_legacy,True
6,paired_net_wins_vs_legacy_positive,True


F1 gain retention: 112.2%
PROCEED TO OFFICIAL TEST


## Block 25. Save the complete validation evidence

### Purpose

This cell packages all validation artefacts so the experiment can be audited, shared and reconstructed without keeping the Colab runtime alive.

### What happens

A timestamped directory is created under `/content`.

The cell saves CSV files containing:

- the fixed respondent sample;
- legacy predictions;
- full-alternative predictions;
- Fast9 predictions;
- target-level comparison tables;
- gain retention;
- bootstrap diagnostics;
- paired win/loss tables;
- feature importance.

It also saves:

- `config.json`;
- `decision.json`;
- relevant Fast9 JSONL cache/checkpoint files.

Finally, the directory is compressed into a ZIP archive.

### Main outputs

- `VALIDATION_SAVE_DIR`;
- `validation_archive`.

### API / cost impact

No API call.

### What to check

The final printed path should point to a ZIP file.

Keep this archive before starting official inference. It is the evidence supporting the model-selection decision and allows metric recalculation without repeating LLM calls.

In [26]:
timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)
VALIDATION_SAVE_DIR = Path(
    f"/content/fast9_validation_{timestamp}"
)
VALIDATION_SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

tables_to_save = {
    "reference_sample": reference_sample,
    "reference_legacy": reference_legacy,
    "reference_full_alternative": reference_full,
    "fast9_validation_result": (
        fast9_validation_result
    ),
    "score_comparison": score_comparison,
    "retention_summary": retention_summary,
    "bootstrap_vs_legacy": (
        bootstrap_vs_legacy
    ),
    "win_loss_vs_legacy": (
        win_loss_vs_legacy
    ),
    "bootstrap_vs_full": (
        bootstrap_vs_full
    ),
    "win_loss_vs_full": (
        win_loss_vs_full
    ),
    "feature_importance": (
        fast9_validation_pipeline
        .feature_importance_table
    ),
}

for name, frame in tables_to_save.items():
    frame.to_csv(
        VALIDATION_SAVE_DIR / f"{name}.csv",
        index=False,
    )

with (
    VALIDATION_SAVE_DIR
    / "config.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        asdict(validation_cfg),
        file,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

with (
    VALIDATION_SAVE_DIR
    / "decision.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "decision": FAST9_DECISION,
            "criteria": {
                key: bool(value)
                for key, value
                in decision_criteria.items()
            },
            "f1_gain_retention": (
                None
                if pd.isna(f1_retention)
                else float(f1_retention)
            ),
        },
        file,
        ensure_ascii=False,
        indent=2,
    )

for source_path in Path(
    "/content"
).glob("*fast9*.jsonl"):
    destination = (
        VALIDATION_SAVE_DIR
        / source_path.name
    )
    if (
        source_path.is_file()
        and source_path.resolve()
        != destination.resolve()
    ):
        shutil.copy2(
            source_path,
            destination,
        )

validation_archive = shutil.make_archive(
    str(VALIDATION_SAVE_DIR),
    "zip",
    VALIDATION_SAVE_DIR,
)

print(
    "Saved:",
    validation_archive,
)


Saved: /content/fast9_validation_20260714_230106.zip


## Block 26. Download the validation archive

### Purpose

This cell transfers the validation ZIP from the temporary Colab runtime to the local computer.

### What happens

`files.download()` opens the browser download flow for `validation_archive`.

### Main outputs

A local ZIP file downloaded through the browser.

### API / cost impact

No LLM call.

### What to check

Confirm that the file has actually downloaded before closing or resetting the Colab runtime. Colab storage is temporary.

In [27]:
from google.colab import files

files.download(
    validation_archive
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Part III. Fit on all training data and predict the official test set

Run this section only after Block 24 prints:

```text
PROCEED TO OFFICIAL TEST
```

The official phase differs from validation in two ways:

1. the statistical pipeline is fitted on all available labelled training respondents;
2. the input is the official test table, so no truth labels or local scores are available.

For 1,050 respondents, the pipeline creates 9,450 primary target-specific LLM jobs. Cache and checkpoints are essential because this is the expensive stage.

## Block 27. Fit the final pipeline on the full training set

### Purpose

After model selection is complete, this cell uses all labelled training data to maximise the information available for official prediction.

### What happens

1. The validated configuration is deep-copied.
2. A separate official cache path is assigned.
3. A new cache-aware Nebius client is created.
4. A new `FastNineCallLLMPipeline` is fitted on the entire `train` table in `mixed` mode.

This recomputes:

- target-specific NMI rankings;
- mRMR selections;
- CatBoost models;
- CatBoost importance;
- hybrid feature plans;
- priors;
- retrieval indices.

The official rankings can differ slightly from validation because more data are now available.

### Main outputs

- `official_cfg`;
- `official_llm`;
- `fast9_official_pipeline`.

### API / cost impact

No LLM calls. This is local statistical fitting.

### What to check

The final message should be:

```text
Full-train pipeline is ready.
```

Do not reuse a validation-fitted pipeline for official inference; doing so would unnecessarily discard labelled training data.

In [28]:
official_cfg = copy.deepcopy(
    CFG
)
official_cfg.cache_path = (
    "llm_cache_fast9_official.jsonl"
)

official_llm = CachedNebiusClient(
    official_cfg,
    api_key=nebius_key,
    client=raw_nebius_client,
)

fast9_official_pipeline = (
    FastNineCallLLMPipeline(
        official_cfg,
        official_llm,
    )
    .fit(
        train,
        mode="mixed",
    )
)

print(
    "Full-train pipeline is ready."
)


Loaded 0 cached LLM responses


Encoding feature columns:   0%|          | 0/278 [00:00<?, ?it/s]

Target feature selection:   0%|          | 0/9 [00:00<?, ?it/s]

Pipeline fit completed in 2.67 minutes.
Full-train pipeline is ready.


## Block 28. Run a two-respondent official preview

### Purpose

Before launching all 9,450 jobs, this cell performs a small operational test on the first two official respondents.

### What happens

The runner creates 18 respondent–target jobs and displays detailed diagnostics:

- final prediction;
- LLM prediction;
- CatBoost prediction;
- response status;
- repair flag;
- probability margin.

Because these exact requests are written to cache, the full official run should reuse them rather than pay for them again.

### Main outputs

- `official_preview_predictions`;
- `official_preview_raw`.

### API / cost impact

Up to 18 paid LLM calls if not already cached.

### What to check

Look for:

- valid predictions for all 18 pairs;
- `success_first_pass` or equivalent status;
- low/zero repair and fallback;
- no obvious one-class collapse;
- exact official label strings;
- reasonable diversity across targets.

If a structural issue appears here, stop before Block 29.

In [29]:
official_preview_predictions, official_preview_raw = (
    run_fast9_predictions(
        pipeline=fast9_official_pipeline,
        input_df=test.head(2),
        run_id="official_preview_2",
        include_truth=False,
    )
)

display(
    official_preview_raw[
        [
            "respondent_id",
            "question_id",
            "prediction",
            "llm_prediction",
            "catboost_prediction",
            "status",
            "repair_used",
            "margin",
        ]
    ]
)


New LLM jobs: 18; already checkpointed: 0


official_preview_2:   0%|          | 0/18 [00:00<?, ?it/s]

Completed 18 predictions in 0.07 minutes.
Repair share: 0.00%
Fallback share: 0.00%


,respondent_id,question_id,prediction,llm_prediction,catboost_prediction,status,repair_used,margin
0,R20070701,Q17,Not so important,Not so important,Not so important,success_first_pass,False,0.492509
1,R20070477,Q209,Might do,Might do,Have done,success_first_pass,False,0.567292
2,R20070701,Q209,Have done,Have done,Have done,success_first_pass,False,0.886094
3,R20070477,Q17,Not so important,Not so important,Not so important,success_first_pass,False,0.361330
4,R20070701,Q201,Monthly,Monthly,Weekly,success_first_pass,False,0.117630
5,R20070477,Q148,A great deal,A great deal,A great deal,success_first_pass,False,0.508979
6,R20070477,Q73,Quite a lot,Quite a lot,Quite a lot,success_first_pass,False,0.602164
7,R20070701,Q73,A great deal,A great deal,A great deal,success_first_pass,False,0.400185
8,R20070477,Q186,Never justifiable,Never justifiable,Sometimes justifiable,success_first_pass,False,0.114840
9,R20070477,Q33,Disagree strongly,Disagree strongly,Disagree,success_first_pass,False,0.338077


## Block 29. Run full official inference

### Purpose

This cell generates the 9,450 predictions evaluated by the leaderboard.

### What happens

The parallel runner processes every official test respondent and every target.

It uses:

- the full-train fitted pipeline;
- the official cache;
- the `official_full` checkpoint;
- up to 48 concurrent workers.

At completion, the cell reports:

- shape of the predictions table;
- coverage;
- repair share;
- fallback share;
- total wall time;
- a preview of the first predictions.

### Main outputs

- `official_predictions` — the three-column submission prediction table;
- `official_raw` — detailed diagnostics for every pair;
- `official_minutes`.

### API / cost impact

Yes. This is the largest paid inference stage.

Cached preview calls and any completed checkpoint jobs are skipped.

### What to check

Expected requirements:

\[
\text{number of predictions}
=
1{,}050 \times 9
=
9{,}450.
\]

Coverage should be exactly 1.0.

Also verify:

- no duplicate respondent–target pairs;
- labels are valid;
- repair and fallback shares are acceptably low;
- the runtime and number of new jobs are plausible.

Do not create the submission ZIP until these checks pass.

In [30]:
official_started = time.perf_counter()

official_predictions, official_raw = (
    run_fast9_predictions(
        pipeline=fast9_official_pipeline,
        input_df=test,
        run_id="official_full",
        include_truth=False,
    )
)

official_minutes = (
    time.perf_counter()
    - official_started
) / 60

print(
    "Predictions:",
    official_predictions.shape,
)
print(
    "Coverage:",
    len(official_predictions)
    / (len(test) * len(target_ids)),
)
print(
    "Repair share:",
    official_raw[
        "repair_used"
    ].mean(),
)
print(
    "Fallback share:",
    official_raw[
        "status"
    ].str.contains(
        "fallback"
    ).mean(),
)
print(
    f"Official wall time: "
    f"{official_minutes:.2f} minutes."
)

display(
    official_predictions.head(20)
)


New LLM jobs: 9,450; already checkpointed: 0


official_full:   0%|          | 0/9450 [00:00<?, ?it/s]

Completed 9,450 predictions in 16.09 minutes.
Repair share: 0.05%
Fallback share: 0.00%
Predictions: (9450, 3)
Coverage: 1.0
Repair share: 0.0005291005291005291
Fallback share: 0.0
Official wall time: 16.10 minutes.


,respondent_id,question_id,prediction
0,R20070701,Q209,Have done
1,R20070701,Q33,Disagree strongly
2,R20070477,Q201,Weekly
3,R20070701,Q227,Not at all often
4,R20070701,Q148,Not at all
5,R20070477,Q148,A great deal
6,R20070701,Q201,Monthly
7,R20070477,Q33,Disagree strongly
8,R20070477,Q73,Quite a lot
9,R20070701,Q242,Not essential


## Block 30. Build the official submission archive

### Purpose

This cell converts the official predictions and method documentation into the required ZIP structure.

### What happens

`write_fast9_submission()` creates:

```text
submission_group6_fast9_llm_primary/
├── predictions.csv
├── features.csv
└── method/
    ├── prompts.jsonl
    └── method.md
```

#### `predictions.csv`

Contains respondent ID, target question ID and exact prediction label.

#### `features.csv`

Discloses target–feature pairs used by:

- the hybrid selected feature set;
- the CatBoost expert.

#### `prompts.jsonl`

Stores one representative reconstructed prompt per target, together with the model name and target-specific feature list.

Prompt reconstruction does not send an API call.

#### `method.md`

Explains:

- one LLM call per target;
- leakage-safe feature selection;
- NMI and mRMR;
- CatBoost importance and prior;
- retrieved analogues;
- 88/12 probability blend;
- repair and fallback;
- parallelism, cache and checkpoints.

Finally, the directory is compressed into a ZIP archive.

### Main outputs

- `write_fast9_submission()`;
- `submission_zip`.

### API / cost impact

No LLM call.

### What to check

Before submission, inspect the ZIP and confirm:

- `predictions.csv` has exactly 9,450 rows;
- expected columns and filenames are present;
- no index column was accidentally saved;
- labels exactly match `targets`;
- `features.csv` contains only permitted variables;
- method files describe the version that actually produced the predictions.

The code in this walkthrough is unchanged from the source notebook, so this block reflects that exact implementation.

In [31]:
def write_fast9_submission(
    predictions: pd.DataFrame,
    pipeline: FastNineCallLLMPipeline,
    run_id: str = "group6_fast9_llm_primary",
) -> Path:
    submission_dir = Path(
        f"submission_{run_id}"
    )
    shutil.rmtree(
        submission_dir,
        ignore_errors=True,
    )
    (
        submission_dir
        / "method"
    ).mkdir(
        parents=True,
        exist_ok=True,
    )

    predictions.to_csv(
        submission_dir
        / "predictions.csv",
        index=False,
    )

    disclosure_rows = []

    for target_id in target_ids:
        used_features = set(
            pipeline.hybrid_selected[
                target_id
            ]
        )
        used_features.update(
            pipeline.expert.features.get(
                target_id,
                [],
            )
        )

        for variable in sorted(
            used_features
        ):
            if variable in candidate_features:
                disclosure_rows.append({
                    "question_id": target_id,
                    "feature_variable_code": variable,
                })

    pd.DataFrame(
        disclosure_rows
    ).drop_duplicates().to_csv(
        submission_dir
        / "features.csv",
        index=False,
    )

    example_row = test.iloc[0]

    with (
        submission_dir
        / "method"
        / "prompts.jsonl"
    ).open(
        "w",
        encoding="utf-8",
    ) as file:
        for target_id in target_ids:
            record = {
                "question_id": target_id,
                "model": (
                    pipeline.config.model
                ),
                "prompt_features": (
                    pipeline.hybrid_selected[
                        target_id
                    ]
                ),
                "prompt": (
                    pipeline.preview_prompt(
                        example_row,
                        target_id,
                    )
                ),
            }
            file.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

    method_text = f"""
Model: {pipeline.config.model}

The method is an LLM-primary target-specific survey-response pipeline.
It makes one LLM call for each of the nine target questions for every
respondent. Feature selection is fit-only and leakage-safe.

For each target, normalized mutual information and mRMR first identify
relevant non-target survey answers. CatBoost is trained on an expanded
feature pool and its feature importance is combined conservatively with
the NMI/mRMR ranking. The first eight mRMR-selected features are protected,
up to four nonlinear CatBoost-important features are added, and a compact
target-specific evidence set is formed.

Each LLM prompt contains up to {pipeline.config.prompt_features} direct
respondent answers, {pipeline.config.n_retrieval_examples} compact labelled
training analogues, a target-specific guide, exact official labels and a
fit-only CatBoost probability prior. Q186 and Q242 are modeled directly in
their official coarsened label spaces.

The LLM returns a probability distribution in JSON. The final distribution
uses 88% LLM probabilities and 12% CatBoost probabilities. There is no
latent-profile call and no ordinary reasoning second pass. A short repair
call is used only when the structured output is invalid. CatBoost is used
as a final fallback after technical failure, ensuring complete coverage.

API calls are parallelized with {pipeline.config.max_workers} workers and
all responses are cached and checkpointed.
""".strip() + "\n"

    (
        submission_dir
        / "method"
        / "method.md"
    ).write_text(
        method_text,
        encoding="utf-8",
    )

    archive_path = shutil.make_archive(
        submission_dir.name,
        "zip",
        root_dir=".",
        base_dir=submission_dir.name,
    )

    return Path(
        archive_path
    )


submission_zip = write_fast9_submission(
    official_predictions,
    fast9_official_pipeline,
)

print(
    "Created:",
    submission_zip,
)


Created: /content/submission_group6_fast9_llm_primary.zip


## Block 31. Download the final submission ZIP

### Purpose

This final cell downloads the generated archive from Colab to the local computer.

### What happens

`files.download()` launches a browser download for `submission_zip`.

### Main outputs

The local submission ZIP ready to send or commit.

### API / cost impact

No LLM call.

### What to check

After downloading:

1. open the ZIP locally;
2. confirm all required files are present;
3. verify the archive is the intended model version;
4. preserve the previous submission separately for rollback and comparison;
5. record the commit hash or submission timestamp for reproducibility.

In [ ]:
from google.colab import files

files.download(
    str(submission_zip)
)
